In [1]:
import pandas as pd
import numpy as np
import copy as cp
import os
import pandas as pd
import random as rd
import math
from sklearn.datasets import load_iris
import matplotlib
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch.nn as nn
import torch 
import torch.nn.functional as F
from torch.autograd import Function 
from scipy import interpolate 
from scipy.stats import norm,entropy
from scipy.interpolate import make_interp_spline
from torch.nn.parameter import Parameter
import seaborn as sns
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import roc_curve,auc
from sklearn.manifold import TSNE

D:\anaconda\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [25]:
iris = load_iris()
X = iris["data"].astype(np.float32) # X为(150,4)的array数组
y = iris["target"].astype(np.int64) # y为标签0,1,2
min_max = MinMaxScaler()
min_max.fit(X)
X = min_max.transform(X)
Ra = rd.randint(75,150)
noise1 = np.random.uniform(low=0,high=0.01,size=(Ra,4))
noise2 = np.random.normal(loc=0.01,scale=0.01,size=(Ra,4))
XX = np.zeros((150,4))
XX[0:Ra,:] = X[0:Ra,:]+noise1+noise2
XX[Ra:,:] = X[Ra:,:]
XX = pd.DataFrame(XX)

In [26]:
XX = np.array(XX)
print(np.shape(XX))
#print(y)
train_ratio = 0.8
index = np.random.permutation(XX.shape[0])
train_index = index[:int(XX.shape[0] * train_ratio)]
test_index = index[int(XX.shape[0] * train_ratio):]
X_train, y_train = XX[train_index], y[train_index]
X_test, y_test = XX[test_index], y[test_index]

(150, 4)


In [27]:
X_tsne = TSNE(n_components=2, random_state=33).fit_transform(X)
X_tsne1 = TSNE(n_components=2, random_state=33).fit_transform(XX)
font = {"color": "darkred",
        "size": 28, 
        "family" : "serif"}

%matplotlib qt5
plt.style.use("seaborn")
plt.figure(figsize=(12, 6))

#matplotlib.rcParams.update({'font.size': 24})
plt.subplot(121) 
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, alpha=0.4, 
            cmap=plt.cm.get_cmap('rainbow', 10))
plt.xticks(fontsize=28)
plt.yticks(fontsize=28)
plt.title("t-SNE", fontdict=font)
cbar = plt.colorbar(ticks=range(10))
cbar.ax.tick_params(labelsize=28)
cbar.set_label(label='iris value', fontdict=font)
plt.clim(-0.5, 9.5)

plt.subplot(122) 
plt.scatter(X_tsne1[:, 0], X_tsne1[:, 1], c=y, alpha=0.4, 
            cmap=plt.cm.get_cmap('rainbow', 10))
plt.xticks(fontsize=28)
plt.yticks(fontsize=28)
plt.title("t-SNE", fontdict=font)
cbar = plt.colorbar(ticks=range(10)) 
cbar.ax.tick_params(labelsize=28)
cbar.set_label(label='iris value with mixed unsymmetrical noise', fontdict=font)
plt.clim(-0.5, 9.5)

C:\Users\15850\AppData\Local\Temp\ipykernel_16572\4283391322.py:8: MatplotlibDeprecationWarning: The seaborn styles shipped by Matplotlib are deprecated since 3.6, as they no longer correspond to the styles shipped by seaborn. However, they will remain available as 'seaborn-v0_8-<style>'. Alternatively, directly use the seaborn API instead.
  plt.style.use("seaborn")
C:\Users\15850\AppData\Local\Temp\ipykernel_16572\4283391322.py:14: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap=plt.cm.get_cmap('rainbow', 10))
C:\Users\15850\AppData\Local\Temp\ipykernel_16572\4283391322.py:25: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap=plt.cm.get_cma

In [7]:
def quantum_normalization(x):
    y = torch.zeros(len(x),len(x[0]))
    if np.shape(x) != (1,len(x[0])):
        x_tot = torch.zeros(len(x))
        for q in range(len(x)):
            for i in range(len(x[0])):
                x_tot[q] = x_tot[q] + x[q][i].item()*x[q][i].item()
            x_tot[q] = torch.sqrt(x_tot[q])
            for j in range(len(x[0])):
                x[q][j] = x[q][j].item()/x_tot[q]
                y[q][j] = x[q][j]

    if np.shape(x) == (1,len(x[0])):
        #print(x)
        #x = x.detach().numpy()
        x_tot1 = torch.zeros(1)
        for i in range(len(x[0])):
            x_tot1 = x_tot1 + x[0][i].item()*x[0][i].item()
        x_tot1 = torch.sqrt(x_tot1)
        #print(x_tot1)
        for j in range(len(x[0])):
            x[0][j] = x[0][j].item()/x_tot1
            y[0][j] = x[0][j]
            
    return y
    
def gate_operation_res(x,n,a,k,b):
    c = torch.zeros(n,2*len(x[0])-3,len(x[0]),len(x[0]))
    b = torch.tensor(np.random.normal(0.01, 0.01, size=(5)), dtype=torch.float32)
    #b = torch.Tensor(5).uniform_(-0.1,0.1)
    for t in range(n):
        if t == 2:
            for i in range(len(x[0])-1):
                iden = torch.eye(len(x[0]))
                iden[i+1][i] = torch.sin(a[t][i])+b[i]
                iden[i][i] = iden[i+1][i+1] = torch.cos(a[t][i])+b[i]
                iden[i][i+1] = -torch.sin(a[t][i])+b[i]
                c[t][i] = iden

            for j in range(len(x[0])-2):
                iden = torch.eye(len(x[0]))
                iden[len(x[0])-2-j][len(x[0])-3-j] = torch.sin(a[t][(j+3)])+b[j+3]
                iden[len(x[0])-2-j][len(x[0])-2-j] = iden[len(x[0])-3-j][len(x[0])-3-j] = torch.cos(a[t][(j+3)])+b[j+3]
                iden[len(x[0])-3-j][len(x[0])-2-j] = -torch.sin(a[t][(j+3)])+b[j+3]
                c[t][j+3] = iden
            
        else: 
            for i in range(len(x[0])-1):
                iden = torch.eye(len(x[0]))
                iden[i+1][i] = torch.sin(a[t][i])
                iden[i][i] = iden[i+1][i+1] = torch.cos(a[t][i])
                iden[i][i+1] = -torch.sin(a[t][i])
                c[t][i] = iden

            for j in range(len(x[0])-2):
                iden = torch.eye(len(x[0]))
                iden[len(x[0])-2-j][len(x[0])-3-j] = torch.sin(a[t][(j+3)])
                iden[len(x[0])-2-j][len(x[0])-2-j] = iden[len(x[0])-3-j][len(x[0])-3-j] = torch.cos(a[t][(j+3)])
                iden[len(x[0])-3-j][len(x[0])-2-j] = -torch.sin(a[t][(j+3)])
                c[t][j+3] = iden
            
   
    L1 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[0][0],c[0][1]),c[0][2]),c[0][3]),c[0][4])
    L2 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[1][0],c[1][1]),c[1][2]),c[1][3]),c[1][4])
    L3 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[2][0],c[2][1]),c[2][2]),c[2][3]),c[2][4])
    L4 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[3][0],c[3][1]),c[3][2]),c[3][3]),c[3][4])
    L5 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[4][0],c[4][1]),c[4][2]),c[4][3]),c[4][4])
    L6 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[5][0],c[5][1]),c[5][2]),c[5][3]),c[5][4])
#    L7 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[6][0],c[6][1]),c[6][2]),c[6][3]),c[6][4])
#    L8 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[7][0],c[7][1]),c[7][2]),c[7][3]),c[7][4])
#    L9 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[8][0],c[8][1]),c[8][2]),c[8][3]),c[8][4])
#    L10 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[9][0],c[9][1]),c[9][2]),c[9][3]),c[9][4])
#    L11 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[10][0],c[10][1]),c[10][2]),c[10][3]),c[10][4])
#    L12 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[11][0],c[11][1]),c[11][2]),c[11][3]),c[11][4])
     
    m1 = torch.matmul(L1,x.T) #列数是batchsize数
    m1.requires_grad_(True)
    m2 = torch.matmul(L2,m1) #列数是batchsize数
    m2 = m2+k[0]*m1
    m2.requires_grad_(True)
   
    m3 = torch.matmul(L3,m2) #列数是batchsize数
    m3 = m3+k[1]*m2
    m3.requires_grad_(True)
    
    m4 = torch.matmul(L4,m3) #列数是batchsize数
    m4 = m4+k[2]*m3
    m4.requires_grad_(True)
    
    m5 = torch.matmul(L5,m4) #列数是batchsize数
    m5 = m5+k[3]*m4
    m5.requires_grad_(True)
   
    m6 = torch.matmul(L6,m5) #列数是batchsize数
    m6 = m6+k[4]*m5
    m6.requires_grad_(True)
    
#    m7 = torch.matmul(L7,m6) #列数是batchsize数
#    m7 = m7+k[5]*m6
#    m7.requires_grad_(True)
    
#    m8 = torch.matmul(L8,m7) #列数是batchsize数
#    m8 = m8+k[6]*m7
#    m8.requires_grad_(True)
    
  #  m9 = torch.matmul(L9,m8) #列数是batchsize数
  #  m9 = m9+k[0]*m8
  #  m9.requires_grad_(True)
    
    return  m6.T  #行数是batchsize

def measurement(x):
    return x

In [30]:
def gate_operation_dense(x,n,a,k,b):
    c = torch.zeros(n,2*len(x[0])-3,len(x[0]),len(x[0]))
    #m1 = torch.eye(len(x[0]))
    #m2 = torch.zeros(2*len(x[0])-3,len(x[0]),len(x[0]))
   # b = torch.tensor(np.random.normal(0.01, 0.01, size=(5,4)), dtype=torch.float32)
    for t in range(n):
        if t == 0:
            for i in range(len(x[0])-1):
                iden = torch.eye(len(x[0]))
                iden[i+1][i] = torch.sin(a[t][i])+b[i]
                iden[i][i] = iden[i+1][i+1] = torch.cos(a[t][i])+b[i]
                iden[i][i+1] = -torch.sin(a[t][i])+b[i]
                c[t][i] = iden

            for j in range(len(x[0])-2):
                iden = torch.eye(len(x[0]))
                iden[len(x[0])-2-j][len(x[0])-3-j] = torch.sin(a[t][(j+3)])+b[j+3]
                iden[len(x[0])-2-j][len(x[0])-2-j] = iden[len(x[0])-3-j][len(x[0])-3-j] = torch.cos(a[t][(j+3)])+b[j+3]
                iden[len(x[0])-3-j][len(x[0])-2-j] = -torch.sin(a[t][(j+3)])+b[j+3]
                c[t][j+3] = iden
            
        else: 
            for i in range(len(x[0])-1):
                iden = torch.eye(len(x[0]))
                iden[i+1][i] = torch.sin(a[t][i])
                iden[i][i] = iden[i+1][i+1] = torch.cos(a[t][i])
                iden[i][i+1] = -torch.sin(a[t][i])
                c[t][i] = iden

            for j in range(len(x[0])-2):
                iden = torch.eye(len(x[0]))
                iden[len(x[0])-2-j][len(x[0])-3-j] = torch.sin(a[t][(j+3)])
                iden[len(x[0])-2-j][len(x[0])-2-j] = iden[len(x[0])-3-j][len(x[0])-3-j] = torch.cos(a[t][(j+3)])
                iden[len(x[0])-3-j][len(x[0])-2-j] = -torch.sin(a[t][(j+3)])
                c[t][j+3] = iden
            
   
    L1 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[0][0],c[0][1]),c[0][2]),c[0][3]),c[0][4])
    L2 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[1][0],c[1][1]),c[1][2]),c[1][3]),c[1][4])
    L3 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[2][0],c[2][1]),c[2][2]),c[2][3]),c[2][4])
    L4 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[3][0],c[3][1]),c[3][2]),c[3][3]),c[3][4])
    L5 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[4][0],c[4][1]),c[4][2]),c[4][3]),c[4][4])
    L6 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[5][0],c[5][1]),c[5][2]),c[5][3]),c[5][4])
#    L7 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[6][0],c[6][1]),c[6][2]),c[6][3]),c[6][4])
#    L8 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[7][0],c[7][1]),c[7][2]),c[7][3]),c[7][4])
#    L9 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[8][0],c[8][1]),c[8][2]),c[8][3]),c[8][4])
#    L10 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[9][0],c[9][1]),c[9][2]),c[9][3]),c[9][4])
#    L11 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[10][0],c[10][1]),c[10][2]),c[10][3]),c[10][4])
#    L12 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[11][0],c[11][1]),c[11][2]),c[11][3]),c[11][4])
     
    m1 = torch.matmul(L1,x.T) #列数是batchsize数
    m1.requires_grad_(True)
    m2 = torch.matmul(L2,m1) #列数是batchsize数
    m2 = m2+k[0]*m1
    m2.requires_grad_(True)
   
    m3 = torch.matmul(L3,m2) #列数是batchsize数
    m3 = m3+k[2]*m2+k[1]*m1
    m3.requires_grad_(True)
    
    m4 = torch.matmul(L4,m3) #列数是batchsize数
    m4 = m4+k[5]*m3+k[4]*m2+k[3]*m1
    m4.requires_grad_(True)
    
    m5 = torch.matmul(L5,m4) #列数是batchsize数
    m5 = m5+k[9]*m4+k[8]*m3+k[7]*m2+k[6]*m1
    m5.requires_grad_(True)
   
    m6 = torch.matmul(L6,m5) #列数是batchsize数
    m6 = m6+k[14]*m5+k[13]*m4+k[12]*m3+k[11]*m2+k[10]*m1
    m6.requires_grad_(True)
    
#    m7 = torch.matmul(L7,m6) #列数是batchsize数
#    m7 = m7+k[20]*m6+k[19]*m5+k[18]*m4+k[17]*m3+k[16]*m2+k[15]*m1
#    m7.requires_grad_(True)
    
#    m8 = torch.matmul(L8,m7) #列数是batchsize数
#    m8 = m8+k[27]*m7+k[26]*m6+k[25]*m5+k[24]*m4+k[23]*m3+k[22]*m2+k[21]*m1
#    m8.requires_grad_(True)
    return  m6.T


In [11]:
class rotate_psi(Function):
    
    @staticmethod
    def forward(ctx,input_,A,a,psi):
        ctx.A = A
        ctx.a = a
        ctx.psi = psi
        y0 = np.sin(a*input_)
        y = A*y0
        x_new = input_*np.cos(psi) - y*np.sin(psi)
        y_new = input_*np.sin(psi) + y*np.cos(psi)
        result = y_new
        ctx.save_for_backward(input_,result)
        return result
    
    @staticmethod    
    def backward(ctx,grad_output): #grad_output = d(loss)/d(result)
        input_,result = ctx.saved_tensors
        return grad_output*(np.sin(ctx.psi)+ctx.A*ctx.a*np.cos(ctx.a*input_)*np.cos(ctx.psi)),None,None,None

def rotation(input_,A,a,psi):
    return rotate_psi().apply(input_,A,a,psi)

In [36]:
#CNN(rot)
Accbox1 = []
Totallossbox1 = []
Accbox2 = []
Totallossbox2 = []
rec1_sam_train = [[],[],[]]
pre1_sam_train = [[],[],[]]
rec1_sam_test = [[],[],[]]
pre1_sam_test = [[],[],[]]
f1 = [[],[]]
f2 = [[],[]]
f3 = [[],[]]
weight1 = [[],[],[]]
weight1_gra = [[],[],[]]


# 定义神经网络模型
class Net1(nn.Module):
    def __init__(self):
        super(Net1, self).__init__()
        #torch.manual_seed(2)
        self.fc1 = nn.Linear(4, 12)
        self.fc2 = nn.Linear(12, 12)
        self.fc3 = nn.Linear(12, 3)

    def forward(self, x):
      #  x = quantum_normalization(x)
        
        #print(self.fc1.weight[0])
        x = self.fc1(x)
        #print(x.shape)
        x = rotation(x,1,0.8,math.pi/4)
        x = self.fc2(x)
        x = rotation(x,1,0.8,math.pi/4)
        self.fc2.weight.data[0][0] = self.fc2.weight.data[0][0]+b[0]
        self.fc2.weight.data[0][1] = self.fc2.weight.data[0][1]+b[1]
        self.fc2.weight.data[0][2] = self.fc2.weight.data[0][2]+b[2]
        self.fc2.weight.data[0][3] = self.fc2.weight.data[0][3]+b[3]
        self.fc2.weight.data[0][4] = self.fc2.weight.data[0][4]+b[4]
        x = self.fc2(x)
        x = rotation(x,1,0.8,math.pi/4)
        x = self.fc2(x)
        x = rotation(x,1,0.8,math.pi/4)
        x = self.fc2(x)
        #print(x.shape)
        x = rotation(x,1.2,1,math.pi/4)
        x = self.fc3(x)
        
        #print(self.fc1.weight)
        return x

# 初始化模型、损失函数和优化器
model = Net1()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# 训练模型
epoch = 300

def train(X_train,y_train):
    eps = 1e-8
    perfect = 0
    Total = 0
    totalloss1 = 0.0
    TP_sam1_train, FP_sam1_train, FN_sam1_train = 0, 0, 0
    TP_sam2_train, FP_sam2_train, FN_sam2_train = 0, 0, 0
    TP_sam3_train, FP_sam3_train, FN_sam3_train = 0, 0, 0
    global Accbox1
    global Totallossbox1
    global rec1_sam_train 
    global pre1_sam_train
    global f1
    global f2
    global f3
    
    X_train = X_train.astype(np.float32)
    inputs = torch.from_numpy(X_train)
    labels = torch.from_numpy(y_train)
    optimizer.zero_grad()
    #print(inputs)
    outputs = model(inputs)
    labels = labels.type(torch.LongTensor)
    #print(labels[0])
    loss = criterion(outputs, labels)
    #print(outputs.shape)
    #print(labels.shape)
    loss.backward()
    optimizer.step()
    predict = torch.argmax(outputs.data, dim=1)
    Total += labels.size(0) #每个iteration中labels.size(0)为batchsize数
    perfect += (predict == labels).sum().item()
    totalloss1 += loss.item()
    Totallossbox1 = np.append(Totallossbox1,totalloss1/len(X_train))
    Accbox1 = np.append(Accbox1, perfect/Total)
    print("Loss on training set: %.4f" % (totalloss1))
    print('Accuracy on training set: %.3f %%' % (100 * perfect/Total)) 
    for i in range(len(labels)):
        if predict[i] == 0 and labels[i] == 0:
            TP_sam1_train += 1
        if predict[i] == 0 and labels[i] != 0:
            FP_sam1_train += 1
        if predict[i] != 0 and labels[i] == 0:
            FN_sam1_train += 1

        if predict[i] == 1 and labels[i] == 1:
            TP_sam2_train += 1
        if predict[i] == 1 and labels[i] != 1:
            FP_sam2_train += 1
        if predict[i] != 1 and labels[i] == 1:
            FN_sam2_train += 1    

        if predict[i] == 2 and labels[i] == 2:
            TP_sam3_train += 1
        if predict[i] == 2 and labels[i] != 2:
            FP_sam3_train += 1
        if predict[i] != 2 and labels[i] == 2:
            FN_sam3_train += 1

            
    #计算3种样本的召回率与精确度
    P1 = TP_sam1_train/(TP_sam1_train+FP_sam1_train+eps)
    R1 = TP_sam1_train/(TP_sam1_train+FN_sam1_train+eps)
    F1_1train = 2*P1*R1/(P1+R1+eps)
    rec1_sam_train[0] = np.append(rec1_sam_train[0],P1)
    pre1_sam_train[0] = np.append(pre1_sam_train[0],R1)
    f1[0] = np.append(f1[0],F1_1train)

    P2 = TP_sam2_train/(TP_sam2_train+FP_sam2_train+eps)
    R2 = TP_sam2_train/(TP_sam2_train+FN_sam2_train+eps)
    F1_2train = 2*P2*R2/(P2+R2+eps)
    rec1_sam_train[1] = np.append(rec1_sam_train[1],P2)
    pre1_sam_train[1] = np.append(pre1_sam_train[1],R2)
    f2[0] = np.append(f2[0],F1_2train)
    
    P3 = TP_sam3_train/(TP_sam3_train+FP_sam3_train+eps)
    R3 = TP_sam3_train/(TP_sam3_train+FN_sam3_train+eps)
    F1_3train = 2*P3*R3/(P3+R3+eps)
    rec1_sam_train[2] = np.append(rec1_sam_train[2],P3)
    pre1_sam_train[2] = np.append(pre1_sam_train[2],R3)
    f3[0] = np.append(f3[0],F1_3train)

    
# 评估模型
def test(X_test,y_test):
    eps = 1e-8
    correct = 0
    total = 0
    totalloss2 = 0.0
    TP_sam1_test, FP_sam1_test, FN_sam1_test = 0, 0, 0
    TP_sam2_test, FP_sam2_test, FN_sam2_test = 0, 0, 0
    TP_sam3_test, FP_sam3_test, FN_sam3_test = 0, 0, 0
    global Accbox2
    global Totallossbox2
    global rec1_sam_test 
    global pre1_sam_test
    global f1
    global f2
    global f3
   
    
    with torch.no_grad():
        X_test = X_test.astype(np.float32)
        inputs = torch.from_numpy(X_test)
        labels = torch.from_numpy(y_test)
        outputs = model(inputs)
        labels = labels.type(torch.LongTensor)
        loss = criterion(outputs,labels)
        predicted = torch.argmax(outputs.data, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        #print(correct)
        totalloss2 += loss.item()
    Totallossbox2 = np.append(Totallossbox2,totalloss2/len(X_test))
    accuracy = (predicted == labels).float().mean() # (predictions == labels):输出值为bool的Tensor
    Accbox2 = np.append(Accbox2, accuracy)
    print("Loss on testing set: %.4f" % (totalloss2))
    print("Accuracy on testing set: %.3f %%" % (accuracy.item() * 100))
    print()
    print()
    print()
    for i in range(len(labels)):
        if predicted[i] == 0 and labels[i] == 0:
            TP_sam1_test += 1
        if predicted[i] == 0 and labels[i] != 0:
            FP_sam1_test += 1
        if predicted[i] != 0 and labels[i] == 0:
            FN_sam1_test += 1

        if predicted[i] == 1 and labels[i] == 1:
            TP_sam2_test += 1
        if predicted[i] == 1 and labels[i] != 1:
            FP_sam2_test += 1
        if predicted[i] != 1 and labels[i] == 1:
            FN_sam2_test += 1    

        if predicted[i] == 2 and labels[i] == 2:
            TP_sam3_test += 1
        if predicted[i] == 2 and labels[i] != 2:
            FP_sam3_test += 1
        if predicted[i] != 2 and labels[i] == 2:
            FN_sam3_test += 1
    
    P1 = TP_sam1_test/(TP_sam1_test+FP_sam1_test+eps)
    R1 = TP_sam1_test/(TP_sam1_test+FN_sam1_test+eps)
    F1_1test = 2*P1*R1/(P1+R1+eps)
    rec1_sam_test[0] = np.append(rec1_sam_test[0],P1)
    pre1_sam_test[0] = np.append(pre1_sam_test[0],R1)
    f1[1] = np.append(f1[1],F1_1test)
    
    P2 = TP_sam2_test/(TP_sam2_test+FP_sam2_test+eps)
    R2 = TP_sam2_test/(TP_sam2_test+FN_sam2_test+eps)
    F1_2test = 2*P2*R2/(P2+R2+eps)
    rec1_sam_test[1] = np.append(rec1_sam_test[1],P2)
    pre1_sam_test[1] = np.append(pre1_sam_test[1],R2)
    f2[1] = np.append(f2[1],F1_2test)
    
    P3 = TP_sam3_test/(TP_sam3_test+FP_sam3_test+eps)
    R3 = TP_sam3_test/(TP_sam3_test+FN_sam3_test+eps)
    F1_3test = 2*P3*R3/(P3+R3+eps)
    rec1_sam_test[2] = np.append(rec1_sam_test[2],P3)
    pre1_sam_test[2] = np.append(pre1_sam_test[2],R3)
    f3[1] = np.append(f3[1],F1_3test)
    #print(f3[1])
    return F.softmax(outputs)

if __name__ == '__main__':
    for i in range(epoch):
        print('epoch',i+1)
        #b = torch.tensor(np.random.normal(0,0.01, size=(5)), dtype=torch.float32)
        b = torch.Tensor(5).uniform_(-0.1,0.01)+torch.tensor(np.random.normal(956,0.01, size=(5)), dtype=torch.float32)
        train(X_train,y_train)
        weight1[0] = np.append(weight1[0],model.fc1.weight[0][0].item())
        weight1[1] = np.append(weight1[1],model.fc1.weight[0][1].item())
        weight1[2] = np.append(weight1[2],model.fc1.weight[0][2].item())
        val1 = test(X_test,y_test)
        if i != 0:
            weight1_gra[0] = np.append(weight1_gra[0],model.fc1.weight.grad[0][0].item())
            weight1_gra[1] = np.append(weight1_gra[1],model.fc1.weight.grad[0][1].item())
            weight1_gra[2] = np.append(weight1_gra[2],model.fc1.weight.grad[0][2].item())
    

epoch 1
Loss on training set: 7776529.5000
Accuracy on training set: 39.167 %
Loss on testing set: 608488783872.0000
Accuracy on testing set: 23.333 %



epoch 2
Loss on training set: 3953939185664.0000
Accuracy on training set: 35.833 %
Loss on testing set: 13616532160512.0000
Accuracy on testing set: 26.667 %



epoch 3
Loss on training set: 34116923817984.0000
Accuracy on training set: 37.500 %
Loss on testing set: 71561453240320.0000
Accuracy on testing set: 33.333 %



epoch 4
Loss on training set: 131028538621952.0000
Accuracy on training set: 37.500 %
Loss on testing set: 223445640019968.0000
Accuracy on testing set: 40.000 %



epoch 5
Loss on training set: 347259615051776.0000
Accuracy on training set: 32.500 %
Loss on testing set: 531378588352512.0000
Accuracy on testing set: 43.333 %





C:\Users\windows\AppData\Local\Temp\ipykernel_10968\2850719850.py:224: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.softmax(outputs)


epoch 6
Loss on training set: 748265645539328.0000
Accuracy on training set: 34.167 %
Loss on testing set: 1063298879455232.0000
Accuracy on testing set: 46.667 %



epoch 7
Loss on training set: 1405702530662400.0000
Accuracy on training set: 45.000 %
Loss on testing set: 1906561484062720.0000
Accuracy on testing set: 50.000 %



epoch 8
Loss on training set: 2414026902470656.0000
Accuracy on training set: 46.667 %
Loss on testing set: 3159224397856768.0000
Accuracy on testing set: 50.000 %



epoch 9
Loss on training set: 3862258736168960.0000
Accuracy on training set: 44.167 %
Loss on testing set: 4908037907152896.0000
Accuracy on testing set: 46.667 %



epoch 10
Loss on training set: 5827768646369280.0000
Accuracy on training set: 40.000 %
Loss on testing set: 7239454603345920.0000
Accuracy on testing set: 43.333 %



epoch 11
Loss on training set: 8414444375769088.0000
Accuracy on training set: 39.167 %
Loss on testing set: 10209236046839808.0000
Accuracy on testing set: 43.333 %

epoch 56
Loss on training set: 23839877401411584.0000
Accuracy on training set: 53.333 %
Loss on testing set: 6501486343225344.0000
Accuracy on testing set: 60.000 %



epoch 57
Loss on training set: 10957436529672192.0000
Accuracy on training set: 59.167 %
Loss on testing set: 1924654436450304.0000
Accuracy on testing set: 70.000 %



epoch 58
Loss on training set: 6961695847088128.0000
Accuracy on training set: 63.333 %
Loss on testing set: 6927529382248448.0000
Accuracy on testing set: 70.000 %



epoch 59
Loss on training set: 15169505588150272.0000
Accuracy on training set: 62.500 %
Loss on testing set: 7828028331655168.0000
Accuracy on testing set: 70.000 %



epoch 60
Loss on training set: 19153473228505088.0000
Accuracy on training set: 60.833 %
Loss on testing set: 686120757100544.0000
Accuracy on testing set: 70.000 %



epoch 61
Loss on training set: 11015513077448704.0000
Accuracy on training set: 61.667 %
Loss on testing set: 20170779182235648.0000
Accuracy on testing set:

Loss on training set: 12562346622844928.0000
Accuracy on training set: 65.833 %
Loss on testing set: 34098273643921408.0000
Accuracy on testing set: 63.333 %



epoch 108
Loss on training set: 28664858694123520.0000
Accuracy on training set: 67.500 %
Loss on testing set: 1220861096886272.0000
Accuracy on testing set: 70.000 %



epoch 109
Loss on training set: 1342111110660096.0000
Accuracy on training set: 65.833 %
Loss on testing set: 48887495311491072.0000
Accuracy on testing set: 63.333 %



epoch 110
Loss on training set: 41293761203929088.0000
Accuracy on training set: 67.500 %
Loss on testing set: 6748480785612800.0000
Accuracy on testing set: 70.000 %



epoch 111
Loss on training set: 7442792313782272.0000
Accuracy on training set: 65.833 %
Loss on testing set: 32459653426184192.0000
Accuracy on testing set: 63.333 %



epoch 112
Loss on training set: 27415925154119680.0000
Accuracy on training set: 67.500 %
Loss on testing set: 27438018465890304.0000
Accuracy on testing set: 

epoch 159
Loss on training set: 76336352127352832.0000
Accuracy on training set: 65.000 %
Loss on testing set: 858612340982022144.0000
Accuracy on testing set: 63.333 %



epoch 160
Loss on training set: 813750342423412736.0000
Accuracy on training set: 65.833 %
Loss on testing set: 615855979048730624.0000
Accuracy on testing set: 63.333 %



epoch 161
Loss on training set: 600716391129022464.0000
Accuracy on training set: 65.833 %
Loss on testing set: 368893469743120384.0000
Accuracy on testing set: 70.000 %



epoch 162
Loss on training set: 474351306455121920.0000
Accuracy on training set: 65.000 %
Loss on testing set: 429448660326023168.0000
Accuracy on testing set: 70.000 %



epoch 163
Loss on training set: 525251479276093440.0000
Accuracy on training set: 65.000 %
Loss on testing set: 95310220560957440.0000
Accuracy on testing set: 70.000 %



epoch 164
Loss on training set: 116145828468359168.0000
Accuracy on training set: 65.000 %
Loss on testing set: 308588899091873792.0000
A

Accuracy on testing set: 70.000 %



epoch 210
Loss on training set: 509865497672548352.0000
Accuracy on training set: 65.000 %
Loss on testing set: 454321743650095104.0000
Accuracy on testing set: 70.000 %



epoch 211
Loss on training set: 530903621877891072.0000
Accuracy on training set: 65.000 %
Loss on testing set: 466273813001142272.0000
Accuracy on testing set: 70.000 %



epoch 212
Loss on training set: 542640736705708032.0000
Accuracy on training set: 65.000 %
Loss on testing set: 472215195880521728.0000
Accuracy on testing set: 70.000 %



epoch 213
Loss on training set: 545615946450993152.0000
Accuracy on training set: 65.000 %
Loss on testing set: 470676051400327168.0000
Accuracy on testing set: 70.000 %



epoch 214
Loss on training set: 537543847475937280.0000
Accuracy on training set: 65.000 %
Loss on testing set: 465505975927832576.0000
Accuracy on testing set: 70.000 %



epoch 215
Loss on training set: 524324350455709696.0000
Accuracy on training set: 65.000 %
Loss on

epoch 261
Loss on training set: 189060440280530944.0000
Accuracy on training set: 45.833 %
Loss on testing set: 213754526547574784.0000
Accuracy on testing set: 40.000 %



epoch 262
Loss on training set: 246110714211598336.0000
Accuracy on training set: 43.333 %
Loss on testing set: 249350201884540928.0000
Accuracy on testing set: 40.000 %



epoch 263
Loss on training set: 291508054673850368.0000
Accuracy on training set: 43.333 %
Loss on testing set: 258673527912136704.0000
Accuracy on testing set: 40.000 %



epoch 264
Loss on training set: 292918831171502080.0000
Accuracy on training set: 43.333 %
Loss on testing set: 255694006019686400.0000
Accuracy on testing set: 43.333 %



epoch 265
Loss on training set: 274724954568654848.0000
Accuracy on training set: 44.167 %
Loss on testing set: 259355345380442112.0000
Accuracy on testing set: 46.667 %



epoch 266
Loss on training set: 253064438062514176.0000
Accuracy on training set: 49.167 %
Loss on testing set: 329172959354683392.0000

In [156]:
# 将label值改为二分类以便画P-R图
label_0 = cp.deepcopy(y_test) 
label_1 = cp.deepcopy(y_test) 
label_2 = cp.deepcopy(y_test) 


for i in range(len(y_test)):
    if label_0[i] != 0:
        label_0[i] = 4   
    else:
        label_0[i] = 0 
    if label_1[i] != 1:
        label_1[i] = 4 
    else:
        label_1[i] = 1 
    if label_2[i] != 2:
        label_2[i] = 4 
    else:
        label_2[i] = 2 

#y_true和y_scores分别是gt label和predict score
y_true0 = np.array(label_0)
y_scores = np.array(val1.numpy()[:,0])
precision0, recall0, thresholds = precision_recall_curve(y_true0, y_scores,pos_label=0)
fpr0, tpr0, thersholds = roc_curve(y_true0, y_scores, pos_label=0)

y_true1 = np.array(label_1)
y_scores = np.array(val1.numpy()[:,1])
precision1, recall1, thresholds = precision_recall_curve(y_true1, y_scores,pos_label=1)
fpr1, tpr1, thersholds = roc_curve(y_true1, y_scores, pos_label=1)

y_true2 = np.array(label_2)
y_scores = np.array(val1.numpy()[:,2])
precision2, recall2, thresholds = precision_recall_curve(y_true2, y_scores,pos_label=2)
fpr2, tpr2, thersholds = roc_curve(y_true2, y_scores, pos_label=2)


pr_auc0,pr_auc1,pr_auc2 = auc(recall0,precision0),auc(recall1,precision1),auc(recall2,precision2)
roc_auc0,roc_auc1,roc_auc2 = auc(fpr0,tpr0),auc(fpr1,tpr1),auc(fpr2,tpr2)


In [23]:
Accbox11 = []
Totallossbox11 = []
Accbox22 = []
Totallossbox22 = []
rec11_sam_train = [[],[],[]]
pre11_sam_train = [[],[],[]]
rec11_sam_test = [[],[],[]]
pre11_sam_test = [[],[],[]]
f11 = [[],[]]
f22 = [[],[]]
f33 = [[],[]]
weight11 = [[],[],[]]
weight11_gra = [[],[],[]]

a = torch.randn(6,5,requires_grad=True)
k = torch.randn(6,requires_grad=True)

# 定义神经网络模型
class Net2(nn.Module):
    def __init__(self):
        super(Net2, self).__init__()
        self.fc = nn.Linear(4, 3)
    def forward(self, x):
        x = quantum_normalization(x)
        x = gate_operation_res(x,6,a,k,b)
        #x = measurement(x)
        x = self.fc(x)
        return x

# 初始化模型、损失函数和优化器
model = Net2()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# 训练模型
def train(X_train,y_train):
    perfect = 0
    Total = 0
    totalloss1 = 0.0
    eps = 1e-8
    global Accbox11
    global Totallossbox11
    TP_sam1_train, FP_sam1_train, FN_sam1_train = 0, 0, 0
    TP_sam2_train, FP_sam2_train, FN_sam2_train = 0, 0, 0
    TP_sam3_train, FP_sam3_train, FN_sam3_train = 0, 0, 0
    global rec11_sam_train 
    global pre11_sam_train
    global f11
    global f22
    global f33
    
    
    inputs = torch.from_numpy(X_train)
    labels = torch.from_numpy(y_train)
    #inputs = X_train
    #labels = y_train
    optimizer.zero_grad()
    #print(inputs)
    outputs = model(inputs)
    labels = labels.type(torch.LongTensor)
    loss = criterion(outputs, labels)
    #print(outputs.shape)
    #print(labels.shape)
    loss.backward()
    optimizer.step()
    predict = torch.argmax(outputs.data, dim=1)
    Total += labels.size(0) #每个iteration中labels.size(0)为batchsize数
    perfect += (predict == labels).sum().item()
    totalloss1 += loss.item()
    Totallossbox11 = np.append(Totallossbox11,totalloss1/len(X_train))
    accuracy = (predict == labels).float().mean() 
    Accbox11 = np.append(Accbox11, perfect/Total)
    print("Loss on training set: %.4f" % (totalloss1))
    print('Accuracy on training set: %.3f %%' % (100 * perfect/Total)) 
    for i in range(len(labels)):
        if predict[i] == 0 and labels[i] == 0:
            TP_sam1_train += 1
        if predict[i] == 0 and labels[i] != 0:
            FP_sam1_train += 1
        if predict[i] != 0 and labels[i] == 0:
            FN_sam1_train += 1

        if predict[i] == 1 and labels[i] == 1:
            TP_sam2_train += 1
        if predict[i] == 1 and labels[i] != 1:
            FP_sam2_train += 1
        if predict[i] != 1 and labels[i] == 1:
            FN_sam2_train += 1    

        if predict[i] == 2 and labels[i] == 2:
            TP_sam3_train += 1
        if predict[i] == 2 and labels[i] != 2:
            FP_sam3_train += 1
        if predict[i] != 2 and labels[i] == 2:
            FN_sam3_train += 1

            
    #计算3种样本的召回率与精确度
    P1 = TP_sam1_train/(TP_sam1_train+FP_sam1_train+eps)
    R1 = TP_sam1_train/(TP_sam1_train+FN_sam1_train+eps)
    F1_1train = 2*P1*R1/(P1+R1+eps)
    rec11_sam_train[0] = np.append(rec11_sam_train[0],P1)
    pre11_sam_train[0] = np.append(pre11_sam_train[0],R1)
    f11[0] = np.append(f11[0],F1_1train)
    
    P2 = TP_sam2_train/(TP_sam2_train+FP_sam2_train+eps)
    R2 = TP_sam2_train/(TP_sam2_train+FN_sam2_train+eps)
    F1_2train = 2*P2*R2/(P2+R2+eps)
    rec11_sam_train[1] = np.append(rec11_sam_train[1],P2)
    pre11_sam_train[1] = np.append(pre11_sam_train[1],R2)
    f22[0] = np.append(f22[0],F1_2train)
    
    P3 = TP_sam3_train/(TP_sam3_train+FP_sam3_train+eps)
    R3 = TP_sam3_train/(TP_sam3_train+FN_sam3_train+eps)
    F1_3train = 2*P3*R3/(P3+R3+eps)
    rec11_sam_train[2] = np.append(rec11_sam_train[2],P3)
    pre11_sam_train[2] = np.append(pre11_sam_train[2],R3)
    f33[0] = np.append(f33[0],F1_3train)

# 评估模型
def test(X_test,y_test):
    correct = 0
    total = 0
    totalloss2 = 0.0
    eps = 1e-8
    global Accbox22
    global Totallossbox22
    TP_sam1_test, FP_sam1_test, FN_sam1_test = 0, 0, 0
    TP_sam2_test, FP_sam2_test, FN_sam2_test = 0, 0, 0
    TP_sam3_test, FP_sam3_test, FN_sam3_test = 0, 0, 0
    global rec11_sam_test 
    global pre11_sam_test
    global f11
    global f22
    global f33
    with torch.no_grad():
        inputs = torch.from_numpy(X_test)
        labels = torch.from_numpy(y_test)
    #    inputs = X_test
    #    labels = y_test
        outputs = model(inputs)
        labels = labels.type(torch.LongTensor)
        loss = criterion(outputs,labels)
        predicted = torch.argmax(outputs.data, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        #print(correct)
        totalloss2 += loss.item()
    Totallossbox22 = np.append(Totallossbox22,totalloss2/len(X_test))
    accuracy = (predicted == labels).float().mean() # (predictions == labels):输出值为bool的Tensor
    Accbox22 = np.append(Accbox22, accuracy)
    print("Loss on testing set: %.4f" % (totalloss2))
    print("Accuracy on testing set: %.3f %%" % (accuracy.item() * 100))
    print()
    print()
    for i in range(len(labels)):
        if predicted[i] == 0 and labels[i] == 0:
            TP_sam1_test += 1
        if predicted[i] == 0 and labels[i] != 0:
            FP_sam1_test += 1
        if predicted[i] != 0 and labels[i] == 0:
            FN_sam1_test += 1

        if predicted[i] == 1 and labels[i] == 1:
            TP_sam2_test += 1
        if predicted[i] == 1 and labels[i] != 1:
            FP_sam2_test += 1
        if predicted[i] != 1 and labels[i] == 1:
            FN_sam2_test += 1    

        if predicted[i] == 2 and labels[i] == 2:
            TP_sam3_test += 1
        if predicted[i] == 2 and labels[i] != 2:
            FP_sam3_test += 1
        if predicted[i] != 2 and labels[i] == 2:
            FN_sam3_test += 1
    
    P1 = TP_sam1_test/(TP_sam1_test+FP_sam1_test+eps)
    R1 = TP_sam1_test/(TP_sam1_test+FN_sam1_test+eps)
    F1_1test = 2*P1*R1/(P1+R1+eps)
    rec11_sam_test[0] = np.append(rec11_sam_test[0],P1)
    pre11_sam_test[0] = np.append(pre11_sam_test[0],R1)
    f11[1] = np.append(f11[1],F1_1test)
    
    P2 = TP_sam2_test/(TP_sam2_test+FP_sam2_test+eps)
    R2 = TP_sam2_test/(TP_sam2_test+FN_sam2_test+eps)
    F1_2test = 2*P2*R2/(P2+R2+eps)
    rec11_sam_test[1] = np.append(rec11_sam_test[1],P2)
    pre11_sam_test[1] = np.append(pre11_sam_test[1],R2)
    f22[1] = np.append(f22[1],F1_2test)
    
    P3 = TP_sam3_test/(TP_sam3_test+FP_sam3_test+eps)
    R3 = TP_sam3_test/(TP_sam3_test+FN_sam3_test+eps)
    F1_3test = 2*P3*R3/(P3+R3+eps)
    rec11_sam_test[2] = np.append(rec11_sam_test[2],P3)
    pre11_sam_test[2] = np.append(pre11_sam_test[2],R3)
    f33[1] = np.append(f33[1],F1_3test)
    #print(f33[1])
    return F.softmax(outputs)

if __name__ == '__main__':
    for i in range(epoch):
        print('epoch',i+1)
        #b = torch.tensor(np.random.normal(0,0.01, size=(5)), dtype=torch.float32)
        b = torch.Tensor(5).uniform_(-0.01,0.1)+torch.tensor(np.random.normal(0.01,0.01, size=(5)), dtype=torch.float32)
        train(X_train,y_train)
        weight11[0] = np.append(weight11[0],a[0][0].item())
        weight11[1] = np.append(weight11[1],a[0][1].item())
        weight11[2] = np.append(weight11[2],a[0][2].item())
        val11 = test(X_test,y_test)    
        if i != 0:
            weight11_gra[0] = np.append(weight11_gra[0],a.grad[0][0].item())
            weight11_gra[1] = np.append(weight11_gra[1],a.grad[0][1].item())
            weight11_gra[2] = np.append(weight11_gra[2],a.grad[0][2].item())  

epoch 1
Loss on training set: 1.1625
Accuracy on training set: 27.500 %
Loss on testing set: 1.1687
Accuracy on testing set: 23.333 %


epoch 2
Loss on training set: 1.1613
Accuracy on training set: 24.167 %


C:\Users\windows\AppData\Local\Temp\ipykernel_10968\4138046303.py:199: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.softmax(outputs)


Loss on testing set: 1.1644
Accuracy on testing set: 20.000 %


epoch 3
Loss on training set: 1.1586
Accuracy on training set: 17.500 %
Loss on testing set: 1.1600
Accuracy on testing set: 10.000 %


epoch 4
Loss on training set: 1.1524
Accuracy on training set: 19.167 %
Loss on testing set: 1.1561
Accuracy on testing set: 10.000 %


epoch 5
Loss on training set: 1.1496
Accuracy on training set: 17.500 %
Loss on testing set: 1.1531
Accuracy on testing set: 10.000 %


epoch 6
Loss on training set: 1.1471
Accuracy on training set: 12.500 %
Loss on testing set: 1.1486
Accuracy on testing set: 3.333 %


epoch 7
Loss on training set: 1.1419
Accuracy on training set: 11.667 %
Loss on testing set: 1.1450
Accuracy on testing set: 3.333 %


epoch 8
Loss on training set: 1.1395
Accuracy on training set: 6.667 %
Loss on testing set: 1.1435
Accuracy on testing set: 3.333 %


epoch 9
Loss on training set: 1.1380
Accuracy on training set: 4.167 %
Loss on testing set: 1.1399
Accuracy on testing set: 

epoch 63
Loss on training set: 0.9848
Accuracy on training set: 78.333 %
Loss on testing set: 0.9853
Accuracy on testing set: 73.333 %


epoch 64
Loss on training set: 0.9927
Accuracy on training set: 75.833 %
Loss on testing set: 0.9814
Accuracy on testing set: 80.000 %


epoch 65
Loss on training set: 0.9877
Accuracy on training set: 80.000 %
Loss on testing set: 0.9826
Accuracy on testing set: 76.667 %


epoch 66
Loss on training set: 0.9830
Accuracy on training set: 76.667 %
Loss on testing set: 0.9764
Accuracy on testing set: 76.667 %


epoch 67
Loss on training set: 0.9770
Accuracy on training set: 78.333 %
Loss on testing set: 0.9726
Accuracy on testing set: 83.333 %


epoch 68
Loss on training set: 0.9757
Accuracy on training set: 72.500 %
Loss on testing set: 0.9733
Accuracy on testing set: 76.667 %


epoch 69
Loss on training set: 0.9769
Accuracy on training set: 80.000 %
Loss on testing set: 0.9673
Accuracy on testing set: 76.667 %


epoch 70
Loss on training set: 0.9747
Acc

epoch 124
Loss on training set: 0.8682
Accuracy on training set: 74.167 %
Loss on testing set: 0.8516
Accuracy on testing set: 76.667 %


epoch 125
Loss on training set: 0.8654
Accuracy on training set: 75.833 %
Loss on testing set: 0.8544
Accuracy on testing set: 73.333 %


epoch 126
Loss on training set: 0.8598
Accuracy on training set: 77.500 %
Loss on testing set: 0.8429
Accuracy on testing set: 80.000 %


epoch 127
Loss on training set: 0.8603
Accuracy on training set: 75.833 %
Loss on testing set: 0.8451
Accuracy on testing set: 76.667 %


epoch 128
Loss on training set: 0.8595
Accuracy on training set: 78.333 %
Loss on testing set: 0.8477
Accuracy on testing set: 73.333 %


epoch 129
Loss on training set: 0.8477
Accuracy on training set: 77.500 %
Loss on testing set: 0.8439
Accuracy on testing set: 73.333 %


epoch 130
Loss on training set: 0.8485
Accuracy on training set: 75.833 %
Loss on testing set: 0.8504
Accuracy on testing set: 73.333 %


epoch 131
Loss on training set: 0.

Loss on training set: 0.7789
Accuracy on training set: 76.667 %
Loss on testing set: 0.7598
Accuracy on testing set: 73.333 %


epoch 185
Loss on training set: 0.7781
Accuracy on training set: 75.833 %
Loss on testing set: 0.7609
Accuracy on testing set: 73.333 %


epoch 186
Loss on training set: 0.7703
Accuracy on training set: 79.167 %
Loss on testing set: 0.7502
Accuracy on testing set: 76.667 %


epoch 187
Loss on training set: 0.7739
Accuracy on training set: 79.167 %
Loss on testing set: 0.7621
Accuracy on testing set: 76.667 %


epoch 188
Loss on training set: 0.7701
Accuracy on training set: 76.667 %
Loss on testing set: 0.7521
Accuracy on testing set: 76.667 %


epoch 189
Loss on training set: 0.7644
Accuracy on training set: 78.333 %
Loss on testing set: 0.7668
Accuracy on testing set: 73.333 %


epoch 190
Loss on training set: 0.7669
Accuracy on training set: 77.500 %
Loss on testing set: 0.7444
Accuracy on testing set: 73.333 %


epoch 191
Loss on training set: 0.7627
Accur

epoch 245
Loss on training set: 0.7065
Accuracy on training set: 76.667 %
Loss on testing set: 0.6942
Accuracy on testing set: 73.333 %


epoch 246
Loss on training set: 0.7057
Accuracy on training set: 76.667 %
Loss on testing set: 0.6953
Accuracy on testing set: 73.333 %


epoch 247
Loss on training set: 0.7092
Accuracy on training set: 77.500 %
Loss on testing set: 0.6932
Accuracy on testing set: 73.333 %


epoch 248
Loss on training set: 0.7126
Accuracy on training set: 77.500 %
Loss on testing set: 0.6908
Accuracy on testing set: 73.333 %


epoch 249
Loss on training set: 0.6991
Accuracy on training set: 77.500 %
Loss on testing set: 0.6939
Accuracy on testing set: 76.667 %


epoch 250
Loss on training set: 0.7016
Accuracy on training set: 76.667 %
Loss on testing set: 0.6952
Accuracy on testing set: 73.333 %


epoch 251
Loss on training set: 0.7028
Accuracy on training set: 77.500 %
Loss on testing set: 0.6804
Accuracy on testing set: 76.667 %


epoch 252
Loss on training set: 0.

In [158]:
#y_true和y_scores分别是gt label和predict score
label_0 = cp.deepcopy(y_test) 
label_1 = cp.deepcopy(y_test) 
label_2 = cp.deepcopy(y_test) 


for i in range(len(y_test)):
    if label_0[i] != 0:
        label_0[i] = 4   
    else:
        label_0[i] = 0 
    if label_1[i] != 1:
        label_1[i] = 4 
    else:
        label_1[i] = 1 
    if label_2[i] != 2:
        label_2[i] = 4 
    else:
        label_2[i] = 2 
        
y_true0 = np.array(label_0)
y_scores = np.array(val11.numpy()[:,0])
precision00, recall00, thresholds = precision_recall_curve(y_true0, y_scores,pos_label=0)
fpr00, tpr00, thersholds = roc_curve(y_true0, y_scores, pos_label=0)

y_true1 = np.array(label_1)
y_scores = np.array(val11.numpy()[:,1])
precision11, recall11, thresholds = precision_recall_curve(y_true1, y_scores,pos_label=1)
fpr11, tpr11, thersholds = roc_curve(y_true1, y_scores, pos_label=1)

y_true2 = np.array(label_2)
y_scores = np.array(val11.numpy()[:,2])
precision22, recall22, thresholds = precision_recall_curve(y_true2, y_scores,pos_label=2)
fpr22, tpr22, thersholds = roc_curve(y_true2, y_scores, pos_label=2)

pr_auc00,pr_auc11,pr_auc22 = auc(recall00,precision00),auc(recall11,precision11),auc(recall22,precision22)
roc_auc00,roc_auc11,roc_auc22 = auc(fpr00,tpr00),auc(fpr11,tpr11),auc(fpr22,tpr22)

In [37]:
Accbox111 = []
Totallossbox111 = []
Accbox222 = []
Totallossbox222 = []
rec111_sam_train = [[],[],[]]
pre111_sam_train = [[],[],[]]
rec111_sam_test = [[],[],[]]
pre111_sam_test = [[],[],[]]
f111 = [[],[]]
f222 = [[],[]]
f333 = [[],[]]
weight111 = [[],[],[]]
weight111_gra = [[],[],[]]

# 定义神经网络模型
class Net3(nn.Module):
    def __init__(self):
        super(Net3, self).__init__()
       # torch.manual_seed(2)
        self.fc1 = nn.Linear(4, 12)
        self.fc2 = nn.Linear(12, 12)
        self.fc3 = nn.Linear(12, 3)

    def forward(self, x):
      #  x = quantum_normalization(x)
       #b = torch.tensor(np.random.normal(0.01,0.01, size=(8,13)), dtype=torch.float32)
        #self.fc1.weight.data = self.fc1.weight.data+noise
        x = self.fc1(x)
        x = F.leaky_relu(x,0.5)
        x = self.fc2(x)
        x = F.leaky_relu(x,0.5)
        self.fc2.weight.data[0][0] = self.fc2.weight.data[0][0]+b[0]
        self.fc2.weight.data[0][1] = self.fc2.weight.data[0][1]+b[1]
        self.fc2.weight.data[0][2] = self.fc2.weight.data[0][2]+b[2]
        self.fc2.weight.data[0][3] = self.fc2.weight.data[0][3]+b[3]
        self.fc2.weight.data[0][4] = self.fc2.weight.data[0][4]+b[4]
        x = self.fc2(x)
        x = F.leaky_relu(x,0.5)
        x = self.fc2(x)
        x = F.leaky_relu(x,0.5)
        x = self.fc2(x)
        x = F.leaky_relu(x,0.5)
        x = self.fc3(x)
        return x

# 初始化模型、损失函数和优化器
model = Net3()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# 训练模型
def train(X_train,y_train):
    perfect = 0
    Total = 0
    totalloss1 = 0.0
    global Accbox111
    global Totallossbox111
    eps = 1e-8
    TP_sam1_train, FP_sam1_train, FN_sam1_train = 0, 0, 0
    TP_sam2_train, FP_sam2_train, FN_sam2_train = 0, 0, 0
    TP_sam3_train, FP_sam3_train, FN_sam3_train = 0, 0, 0
    global rec111_sam_train 
    global pre111_sam_train
    global f111
    global f222
    global f333
    
    X_train = X_train.astype(np.float32)
    inputs = torch.from_numpy(X_train)
    labels = torch.from_numpy(y_train)
    optimizer.zero_grad()
    #print(inputs)
    outputs = model(inputs)
    labels = labels.type(torch.LongTensor)
    loss = criterion(outputs, labels)
    #print(outputs.shape)
    #print(labels.shape)
    loss.backward()
    optimizer.step()
    predict = torch.argmax(outputs.data, dim=1)
    Total += labels.size(0) #每个iteration中labels.size(0)为batchsize数
    perfect += (predict == labels).sum().item()
    totalloss1 += loss.item()
    Totallossbox111 = np.append(Totallossbox111,totalloss1/len(X_train))
    accuracy = (predict == labels).float().mean() 
    Accbox111 = np.append(Accbox111, perfect/Total)
    print("Loss on training set: %.4f" % (totalloss1))
    print('Accuracy on training set: %.3f %%' % (100 * perfect/Total)) 
    for i in range(len(labels)):
        if predict[i] == 0 and labels[i] == 0:
            TP_sam1_train += 1
        if predict[i] == 0 and labels[i] != 0:
            FP_sam1_train += 1
        if predict[i] != 0 and labels[i] == 0:
            FN_sam1_train += 1

        if predict[i] == 1 and labels[i] == 1:
            TP_sam2_train += 1
        if predict[i] == 1 and labels[i] != 1:
            FP_sam2_train += 1
        if predict[i] != 1 and labels[i] == 1:
            FN_sam2_train += 1    

        if predict[i] == 2 and labels[i] == 2:
            TP_sam3_train += 1
        if predict[i] == 2 and labels[i] != 2:
            FP_sam3_train += 1
        if predict[i] != 2 and labels[i] == 2:
            FN_sam3_train += 1

    #计算四种样本的召回率与精确度
    P1 = TP_sam1_train/(TP_sam1_train+FP_sam1_train+eps)
    R1 = TP_sam1_train/(TP_sam1_train+FN_sam1_train+eps)
    F1_1train = 2*P1*R1/(P1+R1+eps)
    rec111_sam_train[0] = np.append(rec111_sam_train[0],P1)
    pre111_sam_train[0] = np.append(pre111_sam_train[0],R1)
    f111[0] = np.append(f111[0],F1_1train)
    
    P2 = TP_sam2_train/(TP_sam2_train+FP_sam2_train+eps)
    R2 = TP_sam2_train/(TP_sam2_train+FN_sam2_train+eps)
    F1_2train = 2*P2*R2/(P2+R2+eps)
    rec111_sam_train[1] = np.append(rec111_sam_train[1],P2)
    pre111_sam_train[1] = np.append(pre111_sam_train[1],R2)
    f222[0] = np.append(f222[0],F1_2train)
    
    P3 = TP_sam3_train/(TP_sam3_train+FP_sam3_train+eps)
    R3 = TP_sam3_train/(TP_sam3_train+FN_sam3_train+eps)
    F1_3train = 2*P3*R3/(P3+R3+eps)
    rec111_sam_train[2] = np.append(rec111_sam_train[2],P3)
    pre111_sam_train[2] = np.append(pre111_sam_train[2],R3)
    f333[0] = np.append(f333[0],F1_3train)
    
    
# 评估模型
def test(X_test,y_test):
    correct = 0
    total = 0
    totalloss2 = 0.0
    global Accbox222
    global Totallossbox222
    eps = 1e-8
    TP_sam1_test, FP_sam1_test, FN_sam1_test = 0, 0, 0
    TP_sam2_test, FP_sam2_test, FN_sam2_test = 0, 0, 0
    TP_sam3_test, FP_sam3_test, FN_sam3_test = 0, 0, 0
    global rec111_sam_test 
    global pre111_sam_test
    global f111
    global f222
    global f333
    
    with torch.no_grad():
        X_test = X_test.astype(np.float32)
        inputs = torch.from_numpy(X_test)
        labels = torch.from_numpy(y_test)
        outputs = model(inputs)
        labels = labels.type(torch.LongTensor)
        loss = criterion(outputs,labels)
        predicted = torch.argmax(outputs.data, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        #print(correct)
        totalloss2 += loss.item()
    Totallossbox222 = np.append(Totallossbox222,totalloss2/len(X_test))
    accuracy = (predicted == labels).float().mean() # (predictions == labels):输出值为bool的Tensor
    Accbox222 = np.append(Accbox222, accuracy)
    print("Loss on testing set: %.4f" % (totalloss2))
    print("Accuracy on testing set: %.3f %%" % (accuracy.item() * 100))
    print()
    print()
    for i in range(len(labels)):
        if predicted[i] == 0 and labels[i] == 0:
            TP_sam1_test += 1
        if predicted[i] == 0 and labels[i] != 0:
            FP_sam1_test += 1
        if predicted[i] != 0 and labels[i] == 0:
            FN_sam1_test += 1

        if predicted[i] == 1 and labels[i] == 1:
            TP_sam2_test += 1
        if predicted[i] == 1 and labels[i] != 1:
            FP_sam2_test += 1
        if predicted[i] != 1 and labels[i] == 1:
            FN_sam2_test += 1    

        if predicted[i] == 2 and labels[i] == 2:
            TP_sam3_test += 1
        if predicted[i] == 2 and labels[i] != 2:
            FP_sam3_test += 1
        if predicted[i] != 2 and labels[i] == 2:
            FN_sam3_test += 1
    
    P1 = TP_sam1_test/(TP_sam1_test+FP_sam1_test+eps)
    R1 = TP_sam1_test/(TP_sam1_test+FN_sam1_test+eps)
    F1_1test = 2*P1*R1/(P1+R1+eps)
    rec111_sam_test[0] = np.append(rec111_sam_test[0],P1)
    pre111_sam_test[0] = np.append(pre111_sam_test[0],R1)
    f111[1] = np.append(f111[1],F1_1test)
    
    P2 = TP_sam2_test/(TP_sam2_test+FP_sam2_test+eps)
    R2 = TP_sam2_test/(TP_sam2_test+FN_sam2_test+eps)
    F1_2test = 2*P2*R2/(P2+R2+eps)
    rec111_sam_test[1] = np.append(rec111_sam_test[1],P2)
    pre111_sam_test[1] = np.append(pre111_sam_test[1],R2)
    f222[1] = np.append(f222[1],F1_2test)
    
    P3 = TP_sam3_test/(TP_sam3_test+FP_sam3_test+eps)
    R3 = TP_sam3_test/(TP_sam3_test+FN_sam3_test+eps)
    F1_3test = 2*P3*R3/(P3+R3+eps)
    rec111_sam_test[2] = np.append(rec111_sam_test[2],P3)
    pre111_sam_test[2] = np.append(pre111_sam_test[2],R3)
    f333[1] = np.append(f333[1],F1_3test)
    #print(f333[1])
    return F.softmax(outputs)

if __name__ == '__main__':
    for i in range(epoch):
        print('epoch',i+1)
        b = torch.tensor(np.random.normal(4564,0.01, size=(5)), dtype=torch.float32)
        #b = torch.Tensor(5).uniform_(-0.01,0.01)+torch.tensor(np.random.normal(2455,0.01, size=(5)), dtype=torch.float32)
        train(X_train,y_train)
        weight111[0] = np.append(weight111[0],model.fc1.weight[0][0].item())
        weight111[1] = np.append(weight111[1],model.fc1.weight[0][1].item())
        weight111[2] = np.append(weight111[2],model.fc1.weight[0][2].item())
        val111 = test(X_test,y_test)    
        if i != 0:
            weight111_gra[0] = np.append(weight111_gra[0],model.fc1.weight.grad[0][0].item())
            weight111_gra[1] = np.append(weight111_gra[1],model.fc1.weight.grad[0][1].item())
            weight111_gra[2] = np.append(weight111_gra[2],model.fc1.weight.grad[0][2].item()) 

epoch 1
Loss on training set: 6493279232.0000
Accuracy on training set: 32.500 %
Loss on testing set: 580620791054336.0000
Accuracy on testing set: 36.667 %


epoch 2
Loss on training set: 4061589778989056.0000
Accuracy on training set: 32.500 %
Loss on testing set: 13024887757078528.0000
Accuracy on testing set: 36.667 %


epoch 3
Loss on training set: 35094143415877632.0000
Accuracy on training set: 32.500 %
Loss on testing set: 68610333027074048.0000
Accuracy on testing set: 36.667 %


epoch 4
Loss on training set: 135046502069305344.0000
Accuracy on training set: 32.500 %
Loss on testing set: 212670940658532352.0000
Accuracy on testing set: 36.667 %


epoch 5
Loss on training set: 356829250206040064.0000
Accuracy on training set: 32.500 %
Loss on testing set: 496868720476422144.0000
Accuracy on testing set: 36.667 %




C:\Users\windows\AppData\Local\Temp\ipykernel_10968\2500872255.py:213: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.softmax(outputs)


epoch 6
Loss on training set: 756205477238407168.0000
Accuracy on training set: 32.500 %
Loss on testing set: 971693813363900416.0000
Accuracy on testing set: 36.667 %


epoch 7
Loss on training set: 1384019194995015680.0000
Accuracy on training set: 32.500 %
Loss on testing set: 1679405330259247104.0000
Accuracy on testing set: 36.667 %


epoch 8
Loss on training set: 2278988124900032512.0000
Accuracy on training set: 32.500 %
Loss on testing set: 2647636918946234368.0000
Accuracy on testing set: 36.667 %


epoch 9
Loss on training set: 3461331598593490944.0000
Accuracy on training set: 32.500 %
Loss on testing set: 3883927793217568768.0000
Accuracy on testing set: 36.667 %


epoch 10
Loss on training set: 4927494894982791168.0000
Accuracy on training set: 32.500 %
Loss on testing set: 5371442505906651136.0000
Accuracy on testing set: 36.667 %


epoch 11
Loss on training set: 6646300399789670400.0000
Accuracy on training set: 32.500 %
Loss on testing set: 7066220383111741440.0000
Accu

Loss on training set: 61479065123746217984.0000
Accuracy on training set: 32.500 %
Loss on testing set: 56005146484863401984.0000
Accuracy on testing set: 33.333 %


epoch 55
Loss on training set: 59709260421352390656.0000
Accuracy on training set: 32.500 %
Loss on testing set: 52535210932904656896.0000
Accuracy on testing set: 33.333 %


epoch 56
Loss on training set: 56765221280958906368.0000
Accuracy on training set: 32.500 %
Loss on testing set: 50093859712637337600.0000
Accuracy on testing set: 26.667 %


epoch 57
Loss on training set: 53094356576093863936.0000
Accuracy on training set: 31.667 %
Loss on testing set: 51144100025349439488.0000
Accuracy on testing set: 26.667 %


epoch 58
Loss on training set: 48845280696413978624.0000
Accuracy on training set: 30.833 %
Loss on testing set: 51194017853250469888.0000
Accuracy on testing set: 26.667 %


epoch 59
Loss on training set: 44651563028169359360.0000
Accuracy on training set: 29.167 %
Loss on testing set: 50358842014931353600.

Loss on testing set: 22584707706332905472.0000
Accuracy on testing set: 56.667 %


epoch 106
Loss on training set: 21067843454885691392.0000
Accuracy on training set: 59.167 %
Loss on testing set: 22391901745332617216.0000
Accuracy on testing set: 56.667 %


epoch 107
Loss on training set: 21362156329362259968.0000
Accuracy on training set: 59.167 %
Loss on testing set: 22662966545951490048.0000
Accuracy on testing set: 56.667 %


epoch 108
Loss on training set: 21514319942553436160.0000
Accuracy on training set: 59.167 %
Loss on testing set: 23418009976838291456.0000
Accuracy on testing set: 56.667 %


epoch 109
Loss on training set: 21558518110966775808.0000
Accuracy on training set: 60.000 %
Loss on testing set: 24275266207666405376.0000
Accuracy on testing set: 56.667 %


epoch 110
Loss on training set: 21717799962436173824.0000
Accuracy on training set: 60.833 %
Loss on testing set: 24800648047789867008.0000
Accuracy on testing set: 56.667 %


epoch 111
Loss on training set: 21805

Loss on training set: 39843126001021222912.0000
Accuracy on training set: 61.667 %
Loss on testing set: 42790182201321324544.0000
Accuracy on testing set: 56.667 %


epoch 157
Loss on training set: 40960449717167194112.0000
Accuracy on training set: 60.833 %
Loss on testing set: 45230447104100401152.0000
Accuracy on testing set: 56.667 %


epoch 158
Loss on training set: 41888301191568293888.0000
Accuracy on training set: 62.500 %
Loss on testing set: 47310454823015415808.0000
Accuracy on testing set: 56.667 %


epoch 159
Loss on training set: 43009970177667235840.0000
Accuracy on training set: 62.500 %
Loss on testing set: 48305825107454984192.0000
Accuracy on testing set: 56.667 %


epoch 160
Loss on training set: 44049101024892289024.0000
Accuracy on training set: 62.500 %
Loss on testing set: 48986976958908727296.0000
Accuracy on testing set: 56.667 %


epoch 161
Loss on training set: 45113023660300435456.0000
Accuracy on training set: 61.667 %
Loss on testing set: 4994331897860875

Loss on training set: 120409875710679711744.0000
Accuracy on training set: 60.833 %
Loss on testing set: 133646236490498310144.0000
Accuracy on testing set: 56.667 %


epoch 207
Loss on training set: 122751536410679836672.0000
Accuracy on training set: 60.833 %
Loss on testing set: 136240696903956692992.0000
Accuracy on testing set: 56.667 %


epoch 208
Loss on training set: 125126806982117818368.0000
Accuracy on training set: 60.833 %
Loss on testing set: 138873446710340747264.0000
Accuracy on testing set: 56.667 %


epoch 209
Loss on training set: 127536540646016811008.0000
Accuracy on training set: 60.833 %
Loss on testing set: 141542911408999497728.0000
Accuracy on testing set: 56.667 %


epoch 210
Loss on training set: 129980367966469881856.0000
Accuracy on training set: 60.833 %
Loss on testing set: 144251668255188451328.0000
Accuracy on testing set: 56.667 %


epoch 211
Loss on training set: 132459054203569963008.0000
Accuracy on training set: 60.833 %
Loss on testing set: 14699

Loss on testing set: 297179908220599140352.0000
Accuracy on testing set: 56.667 %


epoch 253
Loss on training set: 272244567799383457792.0000
Accuracy on training set: 60.833 %
Loss on testing set: 301875121938365415424.0000
Accuracy on testing set: 56.667 %


epoch 254
Loss on training set: 276533173321105145856.0000
Accuracy on training set: 60.833 %
Loss on testing set: 306625223276590268416.0000
Accuracy on testing set: 56.667 %


epoch 255
Loss on training set: 280870421237239644160.0000
Accuracy on training set: 60.833 %
Loss on testing set: 311430775185227120640.0000
Accuracy on testing set: 56.667 %


epoch 256
Loss on training set: 285257613369554239488.0000
Accuracy on training set: 60.833 %
Loss on testing set: 316294170201578012672.0000
Accuracy on testing set: 56.667 %


epoch 257
Loss on training set: 289696632081955684352.0000
Accuracy on training set: 60.833 %
Loss on testing set: 321209215876155310080.0000
Accuracy on testing set: 56.667 %


epoch 258
Loss on training

Loss on training set: 527037335198985355264.0000
Accuracy on training set: 60.833 %
Loss on testing set: 584244344650031169536.0000
Accuracy on testing set: 56.667 %


epoch 300
Loss on training set: 534029982861036093440.0000
Accuracy on training set: 60.833 %
Loss on testing set: 591993737786968506368.0000
Accuracy on testing set: 56.667 %




In [160]:
#y_true和y_scores分别是gt label和predict score
#计算这一组神经网络对于三种样本的p，r值
label_0 = cp.deepcopy(y_test) 
label_1 = cp.deepcopy(y_test) 
label_2 = cp.deepcopy(y_test) 


for i in range(len(y_test)):
    if label_0[i] != 0:
        label_0[i] = 4   
    else:
        label_0[i] = 0 
    if label_1[i] != 1:
        label_1[i] = 4 
    else:
        label_1[i] = 1 
    if label_2[i] != 2:
        label_2[i] = 4 
    else:
        label_2[i] = 2 
        
y_true0 = np.array(label_0)
y_scores = np.array(val111.numpy()[:,0])
precision000, recall000, thresholds = precision_recall_curve(y_true0, y_scores,pos_label=0)
fpr000, tpr000, thersholds = roc_curve(y_true0, y_scores, pos_label=0)

y_true1 = np.array(label_1)
y_scores = np.array(val111.numpy()[:,1])
precision111, recall111, thresholds = precision_recall_curve(y_true1, y_scores,pos_label=1)
fpr111, tpr111, thersholds = roc_curve(y_true1, y_scores, pos_label=1)

y_true2 = np.array(label_2)
y_scores = np.array(val111.numpy()[:,2])
precision222, recall222, thresholds = precision_recall_curve(y_true2, y_scores,pos_label=2)
fpr222, tpr222, thersholds = roc_curve(y_true2, y_scores, pos_label=2)


pr_auc000,pr_auc111,pr_auc222 = auc(recall000,precision000),auc(recall111,precision111),auc(recall222,precision222)
roc_auc000,roc_auc111,roc_auc222 = auc(fpr000,tpr000),auc(fpr111,tpr111),auc(fpr222,tpr222)


In [38]:
Accbox1111 = []
Totallossbox1111 = []
Accbox2222 = []
Totallossbox2222 = []
rec1111_sam_train = [[],[],[]]
pre1111_sam_train = [[],[],[]]
rec1111_sam_test = [[],[],[]]
pre1111_sam_test = [[],[],[]]
f1111 = [[],[]]
f2222 = [[],[]]
f3333 = [[],[]]
weight1111 = [[],[],[]]
weight1111_gra = [[],[],[]]

c = torch.randn(6,5,requires_grad=True)
k = torch.randn(15,requires_grad=True)
# 定义神经网络模型
class Net4(nn.Module):
    def __init__(self):
        super(Net4, self).__init__()
        self.fc = nn.Linear(4, 3)
    def forward(self, x):
        x = quantum_normalization(x)
        x = gate_operation_dense(x,6,c,k,b)
        x = measurement(x)
        x = self.fc(x)
        return x

# 初始化模型、损失函数和优化器
model = Net4()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# 训练模型
def train(X_train,y_train):
    perfect = 0
    Total = 0
    totalloss1 = 0.0
    global Accbox1111
    global Totallossbox1111
    eps = 1e-8
    TP_sam1_train, FP_sam1_train, FN_sam1_train = 0, 0, 0
    TP_sam2_train, FP_sam2_train, FN_sam2_train = 0, 0, 0
    TP_sam3_train, FP_sam3_train, FN_sam3_train = 0, 0, 0
    global rec1111_sam_train 
    global pre1111_sam_train
    global f1111
    global f2222
    global f3333
    
    inputs = torch.from_numpy(X_train)
    labels = torch.from_numpy(y_train)
    optimizer.zero_grad()
    #print(inputs)
    outputs = model(inputs)
    labels = labels.type(torch.LongTensor)
    loss = criterion(outputs, labels)
    #print(outputs.shape)
    #print(labels.shape)
    loss.backward()
    optimizer.step()
    predict = torch.argmax(outputs.data, dim=1)
    Total += labels.size(0) #每个iteration中labels.size(0)为batchsize数
    perfect += (predict == labels).sum().item()
    totalloss1 += loss.item()
    Totallossbox1111 = np.append(Totallossbox1111,totalloss1/len(X_test))
    accuracy = (predict == labels).float().mean() 
    Accbox1111 = np.append(Accbox1111, perfect/Total)
    print("Loss on training set: %.4f" % (totalloss1))
    print('Accuracy on training set: %.3f %%' % (100 * perfect/Total)) 
    for i in range(len(labels)):
        if predict[i] == 0 and labels[i] == 0:
            TP_sam1_train += 1
        if predict[i] == 0 and labels[i] != 0:
            FP_sam1_train += 1
        if predict[i] != 0 and labels[i] == 0:
            FN_sam1_train += 1

        if predict[i] == 1 and labels[i] == 1:
            TP_sam2_train += 1
        if predict[i] == 1 and labels[i] != 1:
            FP_sam2_train += 1
        if predict[i] != 1 and labels[i] == 1:
            FN_sam2_train += 1    

        if predict[i] == 2 and labels[i] == 2:
            TP_sam3_train += 1
        if predict[i] == 2 and labels[i] != 2:
            FP_sam3_train += 1
        if predict[i] != 2 and labels[i] == 2:
            FN_sam3_train += 1

    #计算四种样本的召回率与精确度
    P1 = TP_sam1_train/(TP_sam1_train+FP_sam1_train+eps)
    R1 = TP_sam1_train/(TP_sam1_train+FN_sam1_train+eps)
    F1_1train = 2*P1*R1/(P1+R1+eps)
    rec1111_sam_train[0] = np.append(rec1111_sam_train[0],P1)
    pre1111_sam_train[0] = np.append(pre1111_sam_train[0],R1)
    f1111[0] = np.append(f1111[0],F1_1train)
    
    P2 = TP_sam2_train/(TP_sam2_train+FP_sam2_train+eps)
    R2 = TP_sam2_train/(TP_sam2_train+FN_sam2_train+eps)
    F1_2train = 2*P2*R2/(P2+R2+eps)
    rec1111_sam_train[1] = np.append(rec1111_sam_train[1],P2)
    pre1111_sam_train[1] = np.append(pre1111_sam_train[1],R2)
    f2222[0] = np.append(f2222[0],F1_2train)
    
    P3 = TP_sam3_train/(TP_sam3_train+FP_sam3_train+eps)
    R3 = TP_sam3_train/(TP_sam3_train+FN_sam3_train+eps)
    F1_3train = 2*P3*R3/(P3+R3+eps)
    rec1111_sam_train[2] = np.append(rec1111_sam_train[2],P3)
    pre1111_sam_train[2] = np.append(pre1111_sam_train[2],R3)
    f3333[0] = np.append(f3333[0],F1_3train)
    
    
# 评估模型
def test(X_test,y_test):
    correct = 0
    total = 0
    totalloss2 = 0.0
    global Accbox2222
    global Totallossbox2222
    eps = 1e-8
    TP_sam1_test, FP_sam1_test, FN_sam1_test = 0, 0, 0
    TP_sam2_test, FP_sam2_test, FN_sam2_test = 0, 0, 0
    TP_sam3_test, FP_sam3_test, FN_sam3_test = 0, 0, 0
    global rec1111_sam_test 
    global pre1111_sam_test
    global f1111
    global f2222
    global f3333
    
    with torch.no_grad():
        inputs = torch.from_numpy(X_test)
        labels = torch.from_numpy(y_test)
        outputs = model(inputs)
        labels = labels.type(torch.LongTensor)
        loss = criterion(outputs,labels)
        predicted = torch.argmax(outputs.data, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        #print(correct)
        totalloss2 += loss.item()
    Totallossbox2222 = np.append(Totallossbox2222,totalloss2/len(X_test))
    accuracy = (predicted == labels).float().mean() # (predictions == labels):输出值为bool的Tensor
    Accbox2222 = np.append(Accbox2222, accuracy)
    print("Loss on testing set: %.4f" % (totalloss2))
    print("Accuracy on testing set: %.3f %%" % (accuracy.item() * 100))
    print()
    print()
    for i in range(len(labels)):
        if predicted[i] == 0 and labels[i] == 0:
            TP_sam1_test += 1
        if predicted[i] == 0 and labels[i] != 0:
            FP_sam1_test += 1
        if predicted[i] != 0 and labels[i] == 0:
            FN_sam1_test += 1

        if predicted[i] == 1 and labels[i] == 1:
            TP_sam2_test += 1
        if predicted[i] == 1 and labels[i] != 1:
            FP_sam2_test += 1
        if predicted[i] != 1 and labels[i] == 1:
            FN_sam2_test += 1    

        if predicted[i] == 2 and labels[i] == 2:
            TP_sam3_test += 1
        if predicted[i] == 2 and labels[i] != 2:
            FP_sam3_test += 1
        if predicted[i] != 2 and labels[i] == 2:
            FN_sam3_test += 1

    
    P1 = TP_sam1_test/(TP_sam1_test+FP_sam1_test+eps)
    R1 = TP_sam1_test/(TP_sam1_test+FN_sam1_test+eps)
    F1_1test = 2*P1*R1/(P1+R1+eps)
    rec1111_sam_test[0] = np.append(rec1111_sam_test[0],P1)
    pre1111_sam_test[0] = np.append(pre1111_sam_test[0],R1)
    f1111[1] = np.append(f1111[1],F1_1test)
    
    P2 = TP_sam2_test/(TP_sam2_test+FP_sam2_test+eps)
    R2 = TP_sam2_test/(TP_sam2_test+FN_sam2_test+eps)
    F1_2test = 2*P2*R2/(P2+R2+eps)
    rec1111_sam_test[1] = np.append(rec1111_sam_test[1],P2)
    pre1111_sam_test[1] = np.append(pre1111_sam_test[1],R2)
    f2222[1] = np.append(f2222[1],F1_2test)
    
    P3 = TP_sam3_test/(TP_sam3_test+FP_sam3_test+eps)
    R3 = TP_sam3_test/(TP_sam3_test+FN_sam3_test+eps)
    F1_3test = 2*P3*R3/(P3+R3+eps)
    rec1111_sam_test[2] = np.append(rec1111_sam_test[2],P3)
    pre1111_sam_test[2] = np.append(pre1111_sam_test[2],R3)
    f3333[1] = np.append(f3333[1],F1_3test)
    #print(f3333[1])
    return F.softmax(outputs)

if __name__ == '__main__':
    for i in range(epoch):
        print('epoch',i+1)
        #b = torch.tensor(np.random.normal(0,0.01, size=(5)), dtype=torch.float32)
        b = torch.Tensor(5).uniform_(-0.01,0.1)+torch.tensor(np.random.normal(0.01,0.01, size=(5)), dtype=torch.float32)
        train(X_train,y_train)
        weight1111[0] = np.append(weight1111[0],a[0][0].item())
        weight1111[1] = np.append(weight1111[1],a[0][1].item())
        weight1111[2] = np.append(weight1111[2],a[0][2].item())
        val1111 = test(X_test,y_test)    
        if i != 0:
            weight1111_gra[0] = np.append(weight1111_gra[0],c.grad[0][0].item())
            weight1111_gra[1] = np.append(weight1111_gra[1],c.grad[0][1].item())
            weight1111_gra[2] = np.append(weight1111_gra[2],c.grad[0][2].item()) 

epoch 1
Loss on training set: 1.4427
Accuracy on training set: 4.167 %
Loss on testing set: 1.5083
Accuracy on testing set: 6.667 %


epoch 2
Loss on training set: 1.4422
Accuracy on training set: 7.500 %


C:\Users\windows\AppData\Local\Temp\ipykernel_10968\854708751.py:195: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.softmax(outputs)


Loss on testing set: 1.5130
Accuracy on testing set: 10.000 %


epoch 3
Loss on training set: 1.4097
Accuracy on training set: 8.333 %
Loss on testing set: 1.4729
Accuracy on testing set: 13.333 %


epoch 4
Loss on training set: 1.3648
Accuracy on training set: 23.333 %
Loss on testing set: 1.4203
Accuracy on testing set: 26.667 %


epoch 5
Loss on training set: 1.3442
Accuracy on training set: 30.000 %
Loss on testing set: 1.4013
Accuracy on testing set: 30.000 %


epoch 6
Loss on training set: 1.3579
Accuracy on training set: 31.667 %
Loss on testing set: 1.4231
Accuracy on testing set: 30.000 %


epoch 7
Loss on training set: 1.3106
Accuracy on training set: 33.333 %
Loss on testing set: 1.3720
Accuracy on testing set: 30.000 %


epoch 8
Loss on training set: 1.2807
Accuracy on training set: 34.167 %
Loss on testing set: 1.3372
Accuracy on testing set: 30.000 %


epoch 9
Loss on training set: 1.2718
Accuracy on training set: 34.167 %
Loss on testing set: 1.3265
Accuracy on testing s

Loss on training set: 0.6974
Accuracy on training set: 68.333 %
Loss on testing set: 0.6999
Accuracy on testing set: 63.333 %


epoch 64
Loss on training set: 0.6810
Accuracy on training set: 67.500 %
Loss on testing set: 0.6818
Accuracy on testing set: 63.333 %


epoch 65
Loss on training set: 0.6635
Accuracy on training set: 70.000 %
Loss on testing set: 0.6624
Accuracy on testing set: 66.667 %


epoch 66
Loss on training set: 0.6686
Accuracy on training set: 68.333 %
Loss on testing set: 0.6640
Accuracy on testing set: 60.000 %


epoch 67
Loss on training set: 0.6466
Accuracy on training set: 70.833 %
Loss on testing set: 0.6400
Accuracy on testing set: 60.000 %


epoch 68
Loss on training set: 0.6503
Accuracy on training set: 70.000 %
Loss on testing set: 0.6401
Accuracy on testing set: 56.667 %


epoch 69
Loss on training set: 0.6589
Accuracy on training set: 70.000 %
Loss on testing set: 0.6522
Accuracy on testing set: 60.000 %


epoch 70
Loss on training set: 0.6536
Accuracy on 

Loss on training set: 0.5664
Accuracy on training set: 71.667 %
Loss on testing set: 0.5566
Accuracy on testing set: 56.667 %


epoch 124
Loss on training set: 0.5669
Accuracy on training set: 70.833 %
Loss on testing set: 0.5577
Accuracy on testing set: 63.333 %


epoch 125
Loss on training set: 0.5602
Accuracy on training set: 68.333 %
Loss on testing set: 0.5385
Accuracy on testing set: 76.667 %


epoch 126
Loss on training set: 0.5393
Accuracy on training set: 73.333 %
Loss on testing set: 0.5297
Accuracy on testing set: 66.667 %


epoch 127
Loss on training set: 0.5607
Accuracy on training set: 70.833 %
Loss on testing set: 0.5476
Accuracy on testing set: 60.000 %


epoch 128
Loss on training set: 0.5675
Accuracy on training set: 70.000 %
Loss on testing set: 0.5599
Accuracy on testing set: 60.000 %


epoch 129
Loss on training set: 0.5292
Accuracy on training set: 71.667 %
Loss on testing set: 0.5234
Accuracy on testing set: 63.333 %


epoch 130
Loss on training set: 0.5608
Accur

Loss on training set: 0.5183
Accuracy on training set: 70.833 %
Loss on testing set: 0.5146
Accuracy on testing set: 63.333 %


epoch 184
Loss on training set: 0.5179
Accuracy on training set: 71.667 %
Loss on testing set: 0.5069
Accuracy on testing set: 73.333 %


epoch 185
Loss on training set: 0.5151
Accuracy on training set: 70.833 %
Loss on testing set: 0.5106
Accuracy on testing set: 63.333 %


epoch 186
Loss on training set: 0.5156
Accuracy on training set: 70.833 %
Loss on testing set: 0.5080
Accuracy on testing set: 60.000 %


epoch 187
Loss on training set: 0.5123
Accuracy on training set: 72.500 %
Loss on testing set: 0.5131
Accuracy on testing set: 60.000 %


epoch 188
Loss on training set: 0.5099
Accuracy on training set: 70.833 %
Loss on testing set: 0.5161
Accuracy on testing set: 60.000 %


epoch 189
Loss on training set: 0.4985
Accuracy on training set: 72.500 %
Loss on testing set: 0.4966
Accuracy on testing set: 66.667 %


epoch 190
Loss on training set: 0.5058
Accur

epoch 244
Loss on training set: 0.4927
Accuracy on training set: 69.167 %
Loss on testing set: 0.4844
Accuracy on testing set: 73.333 %


epoch 245
Loss on training set: 0.4885
Accuracy on training set: 72.500 %
Loss on testing set: 0.4855
Accuracy on testing set: 73.333 %


epoch 246
Loss on training set: 0.4962
Accuracy on training set: 70.833 %
Loss on testing set: 0.4847
Accuracy on testing set: 73.333 %


epoch 247
Loss on training set: 0.4921
Accuracy on training set: 70.833 %
Loss on testing set: 0.4939
Accuracy on testing set: 60.000 %


epoch 248
Loss on training set: 0.4903
Accuracy on training set: 72.500 %
Loss on testing set: 0.4949
Accuracy on testing set: 60.000 %


epoch 249
Loss on training set: 0.4996
Accuracy on training set: 70.833 %
Loss on testing set: 0.4929
Accuracy on testing set: 60.000 %


epoch 250
Loss on training set: 0.4934
Accuracy on training set: 70.833 %
Loss on testing set: 0.4954
Accuracy on testing set: 63.333 %


epoch 251
Loss on training set: 0.

In [162]:
#第四种神经网络三种样本的P,R值
label_0 = cp.deepcopy(y_test) 
label_1 = cp.deepcopy(y_test) 
label_2 = cp.deepcopy(y_test) 


for i in range(len(y_test)):
    if label_0[i] != 0:
        label_0[i] = 4   
    else:
        label_0[i] = 0 
    if label_1[i] != 1:
        label_1[i] = 4 
    else:
        label_1[i] = 1 
    if label_2[i] != 2:
        label_2[i] = 4 
    else:
        label_2[i] = 2 
        
y_true0 = np.array(label_0)
y_scores = np.array(val1111.numpy()[:,0])
precision0000, recall0000, thresholds = precision_recall_curve(y_true0, y_scores,pos_label=0)
fpr0000, tpr0000, thersholds = roc_curve(y_true0, y_scores, pos_label=0)

y_true1 = np.array(label_1)
y_scores = np.array(val1111.numpy()[:,1])
precision1111, recall1111, thresholds = precision_recall_curve(y_true1, y_scores,pos_label=1)
fpr1111, tpr1111, thersholds = roc_curve(y_true1, y_scores, pos_label=1)

y_true2 = np.array(label_2)
y_scores = np.array(val1111.numpy()[:,2])
precision2222, recall2222, thresholds = precision_recall_curve(y_true2, y_scores,pos_label=2)
fpr2222, tpr2222, thersholds = roc_curve(y_true2, y_scores, pos_label=2)

pr_auc0000,pr_auc1111,pr_auc2222 = auc(recall0000,precision0000),auc(recall1111,precision1111),auc(recall2222,precision2222)
roc_auc0000,roc_auc1111,roc_auc2222 = auc(fpr0000,tpr0000),auc(fpr1111,tpr1111),auc(fpr2222,tpr2222)


In [145]:
pr_aucFNN1 = (pr_auc0+pr_auc1+pr_auc2)/3
pr_aucQRFNN = (pr_auc00+pr_auc11+pr_auc22)/3
pr_aucFNN2 = (pr_auc000+pr_auc111+pr_auc222)/3
pr_aucQDFNN = (pr_auc0000+pr_auc1111+pr_auc2222)/3

roc_aucFNN1 = (roc_auc0+roc_auc1+roc_auc2)/3
roc_aucQRFNN = (roc_auc00+roc_auc11+roc_auc22)/3
roc_aucFNN2 = (roc_auc000+roc_auc111+roc_auc222)/3
roc_aucQDFNN = (roc_auc0000+roc_auc1111+roc_auc2222)/3
print('4种神经网络P-R曲线面积为:',pr_aucFNN1,pr_aucQRFNN,pr_aucFNN2,pr_aucQDFNN)
print('4种神经网络ROC曲线面积为:',roc_aucFNN1,roc_aucQRFNN,roc_aucFNN2,roc_aucQDFNN)

%matplotlib qt5
plt.subplot(221)
plt.plot(recall0,precision0,label = "P-R curve of FNN(rot-relu)",lw = 3)
plt.plot(recall00,precision00,label = "P-R curve of QRFNN",lw = 3)
plt.plot(recall000,precision000,label = "P-R curve of FNN(leaky-relu)",lw = 3)
plt.plot(recall0000,precision0000,label = "P-R curve of QDFNN",lw = 3)
plt.title("precision-recall plot of iris of sample 0")
plt.xlabel("recall")
plt.ylabel("precision")
plt.xlim(0,1.1)
plt.ylim(0,1.1)
plt.legend(loc='lower right')
plt.grid()

plt.subplot(222)
plt.plot(recall1,precision1,label = "P-R curve of FNN(rot-relu)",lw = 3)
plt.plot(recall11,precision11,label = "P-R curve of QRFNN",lw = 3)
plt.plot(recall111,precision111,label = "P-R curve of FNN(leaky-relu)",lw = 3)
plt.plot(recall1111,precision1111,label = "P-R curve of QDFNN",lw = 3)
plt.title("precision-recall plot of iris of sample 1")
plt.xlabel("recall")
plt.ylabel("precision")
plt.xlim(0,1.1)
plt.ylim(0,1.1)
plt.legend(loc='lower right')
plt.grid()

plt.subplot(223)
plt.plot(recall2,precision2,label = "P-R curve of FNN(rot-relu)",lw = 3)
plt.plot(recall22,precision22,label = "P-R curve of QRFNN",lw = 3)
plt.plot(recall222,precision222,label = "P-R curve of FNN(leaky-relu)",lw = 3)
plt.plot(recall2222,precision2222,label = "P-R curve of QDFNN",lw = 3)
plt.title("precision-recall plot of iris of sample 2")
plt.xlabel("recall")
plt.ylabel("precision")
plt.xlim(0,1.1)
plt.ylim(0,1.1)
plt.legend(loc='lower right')
plt.grid()

4种神经网络P-R曲线面积为: 0.9999999999999999 0.702268454649407 0.7520811125969855 0.7266669654566481
4种神经网络ROC曲线面积为: 1.0 0.8704087109884212 0.9203281956905145 0.8999309868875086


In [164]:
%matplotlib qt5
plt.subplot(221)
plt.plot(fpr0,tpr0,label = "ROC curve of FNN(rot-relu)",lw = 3)
plt.plot(fpr00,tpr00,label = "ROC curve of QRFNN",lw = 3)
plt.plot(fpr000,tpr000,label = "ROC curve of FNN(leaky-relu)",lw = 3)
plt.plot(fpr0000,tpr0000,label = "ROC curve of QDFNN",lw = 3)
plt.title("ROC plot of iris of sample 0")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.xlim(0,1.1)
plt.ylim(0,1.1)
plt.style.use('classic')
plt.legend(loc='lower right')
plt.grid()

plt.subplot(222)
plt.plot(fpr1,tpr1,label = "ROC curve of FNN(rot-relu)",lw = 3)
plt.plot(fpr11,tpr11,label = "ROC curve of QRFNN",lw = 3)
plt.plot(fpr111,tpr111,label = "ROC curve of FNN(leaky-relu)",lw = 3)
plt.plot(fpr1111,tpr1111,label = "ROC curve of QDFNN",lw = 3)
plt.title("ROC plot of iris of sample 1")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.style.use('classic')
plt.xlim(0,1.1)
plt.ylim(0,1.1)
plt.legend(loc='lower right')
plt.grid()

plt.subplot(223)
plt.plot(fpr2,tpr2,label = "ROC curve of FNN(rot-relu)",lw = 3)
plt.plot(fpr22,tpr22,label = "ROC curve of QRFNN",lw = 3)
plt.plot(fpr222,tpr222,label = "ROC curve of FNN(leaky-relu)",lw = 3)
plt.plot(fpr2222,tpr2222,label = "ROC curve of QDFNN",lw = 3)
plt.title("ROC plot of iris of sample 2")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.style.use('classic')
plt.xlim(0,1.1)
plt.ylim(0,1.1)
plt.legend(loc='lower right')
plt.grid()


In [40]:
x1 = np.linspace(1,epoch,epoch)
x2 = np.linspace(2,epoch,epoch-1)
xs = np.linspace(1,epoch,1000000)
xss = np.linspace(2,epoch,1000000)
plt.figure(figsize = (18,6))

#rec召回率，pre精确度，前面的数字代表第几类样本

fitacc1 = make_interp_spline(x1,Accbox1)
fitacc2 = make_interp_spline(x1,Accbox2)

ys3 = fitacc1(xs)
ys4 = fitacc2(xs)

fitacc11 = make_interp_spline(x1,Accbox11)
fitacc22 = make_interp_spline(x1,Accbox22)

ys33 = fitacc11(xs)
ys44 = fitacc22(xs)

fitacc111 = make_interp_spline(x1,Accbox111)
fitacc222 = make_interp_spline(x1,Accbox222)

ys333 = fitacc111(xs)
ys444 = fitacc222(xs)

fitacc111e = make_interp_spline(x1,Accbox1111)
fitacc222e = make_interp_spline(x1,Accbox2222)

ys333e = fitacc111e(xs)
ys444e = fitacc222e(xs)

In [165]:
x1 = np.linspace(1,epoch,epoch)
x2 = np.linspace(2,epoch,epoch-1)
xs = np.linspace(1,epoch,1000000)
xss = np.linspace(2,epoch,1000000)
plt.figure(figsize = (18,6))

#rec召回率，pre精确度，前面的数字代表第几类样本
fitloss1 = make_interp_spline(x1,Totallossbox1)
fitloss2 = make_interp_spline(x1,Totallossbox2)
fitacc1 = make_interp_spline(x1,Accbox1)
fitacc2 = make_interp_spline(x1,Accbox2)
fitrec1_sam1_train = make_interp_spline(x1,rec1_sam_train[0])
fitrec1_sam2_train = make_interp_spline(x1,rec1_sam_train[1])
fitrec1_sam3_train = make_interp_spline(x1,rec1_sam_train[2])
fitpre1_sam1_train = make_interp_spline(x1,pre1_sam_train[0])
fitpre1_sam2_train = make_interp_spline(x1,pre1_sam_train[1])
fitpre1_sam3_train = make_interp_spline(x1,pre1_sam_train[2])
fitrec1_sam1_test = make_interp_spline(x1,rec1_sam_test[0])
fitrec1_sam2_test = make_interp_spline(x1,rec1_sam_test[1])
fitrec1_sam3_test = make_interp_spline(x1,rec1_sam_test[2])
fitpre1_sam1_test = make_interp_spline(x1,pre1_sam_test[0])
fitpre1_sam2_test = make_interp_spline(x1,pre1_sam_test[1])
fitpre1_sam3_test = make_interp_spline(x1,pre1_sam_test[2])
#这里默认是F1-score,数字代表第几类样本
fit_f1_train = make_interp_spline(x1,f1[0])
fit_f1_test = make_interp_spline(x1,f1[1])
fit_f2_train = make_interp_spline(x1,f2[0])
fit_f2_test = make_interp_spline(x1,f2[1])
fit_f3_train = make_interp_spline(x1,f3[0])
fit_f3_test = make_interp_spline(x1,f3[1])
fit_weight1 = make_interp_spline(x1,weight1[0])
fit_weight2 = make_interp_spline(x1,weight1[1])
fit_weight3 = make_interp_spline(x1,weight1[2])
fit_weight_gra1 = make_interp_spline(x2,weight1_gra[0])
fit_weight_gra2 = make_interp_spline(x2,weight1_gra[1])
fit_weight_gra3 = make_interp_spline(x2,weight1_gra[2])
ys1 = fitloss1(xs)
ys2 = fitloss2(xs)
ys3 = fitacc1(xs)
ys4 = fitacc2(xs)
ys5 = fitrec1_sam1_train(xs)
ys6 = fitrec1_sam2_train(xs)
ys7 = fitrec1_sam3_train(xs)
ys9 = fitpre1_sam1_train(xs)
ys10 = fitpre1_sam2_train(xs)
ys11 = fitpre1_sam3_train(xs)
ys13 = fitrec1_sam1_test(xs)
ys14 = fitrec1_sam2_test(xs)
ys15 = fitrec1_sam3_test(xs)
ys17 = fitpre1_sam1_test(xs)
ys18 = fitpre1_sam2_test(xs)
ys19 = fitpre1_sam3_test(xs)
ys21 = fit_f1_train(xs)
ys22 = fit_f1_test(xs)
ys23 = fit_f2_train(xs)
ys24 = fit_f2_test(xs)
ys25 = fit_f3_train(xs)
ys26 = fit_f3_test(xs)
ys29 = fit_weight1(xs)
ys30 = fit_weight2(xs)
ys31 = fit_weight3(xs)
ys32 = fit_weight_gra1(xss)
ys33 = fit_weight_gra2(xss)
ys34 = fit_weight_gra3(xss)




fitloss11 = make_interp_spline(x1,Totallossbox11)
fitloss22 = make_interp_spline(x1,Totallossbox22)
fitacc11 = make_interp_spline(x1,Accbox11)
fitacc22 = make_interp_spline(x1,Accbox22)
fitrec11_sam1_train = make_interp_spline(x1,rec11_sam_train[0])
fitrec11_sam2_train = make_interp_spline(x1,rec11_sam_train[1])
fitrec11_sam3_train = make_interp_spline(x1,rec11_sam_train[2])
fitpre11_sam1_train = make_interp_spline(x1,pre11_sam_train[0])
fitpre11_sam2_train = make_interp_spline(x1,pre11_sam_train[1])
fitpre11_sam3_train = make_interp_spline(x1,pre11_sam_train[2])
fitrec11_sam1_test = make_interp_spline(x1,rec11_sam_test[0])
fitrec11_sam2_test = make_interp_spline(x1,rec11_sam_test[1])
fitrec11_sam3_test = make_interp_spline(x1,rec11_sam_test[2])
fitpre11_sam1_test = make_interp_spline(x1,pre11_sam_test[0])
fitpre11_sam2_test = make_interp_spline(x1,pre11_sam_test[1])
fitpre11_sam3_test = make_interp_spline(x1,pre11_sam_test[2])
fit_f11_train = make_interp_spline(x1,f11[0])
fit_f11_test = make_interp_spline(x1,f11[1])
fit_f22_train = make_interp_spline(x1,f22[0])
fit_f22_test = make_interp_spline(x1,f22[1])
fit_f33_train = make_interp_spline(x1,f33[0])
fit_f33_test = make_interp_spline(x1,f33[1])
fit_weight11 = make_interp_spline(x1,weight11[0])
fit_weight22 = make_interp_spline(x1,weight11[1])
fit_weight33 = make_interp_spline(x1,weight11[2])
fit_weight_gra11 = make_interp_spline(x2,weight11_gra[0])
fit_weight_gra22 = make_interp_spline(x2,weight11_gra[1])
fit_weight_gra33 = make_interp_spline(x2,weight11_gra[2])
yss11 = fitloss11(xs)
yss22 = fitloss22(xs)
ys33 = fitacc11(xs)
ys44 = fitacc22(xs)
ys55 = fitrec11_sam1_train(xs)
ys66 = fitrec11_sam2_train(xs)
ys77 = fitrec11_sam3_train(xs)
ys99 = fitpre11_sam1_train(xs)
ys100 = fitpre11_sam2_train(xs)
yss111 = fitpre11_sam3_train(xs)
ys133 = fitrec11_sam1_test(xs)
ys144 = fitrec11_sam2_test(xs)
ys155 = fitrec11_sam3_test(xs)
ys177 = fitpre11_sam1_test(xs)
ys188 = fitpre11_sam2_test(xs)
ys199 = fitpre11_sam3_test(xs)
ys211 = fit_f11_train(xs)
yss222 = fit_f11_test(xs)
ys233 = fit_f22_train(xs)
ys244 = fit_f22_test(xs)
ys255 = fit_f33_train(xs)
ys266 = fit_f33_test(xs)
ys299 = fit_weight11(xs)
ys300 = fit_weight22(xs)
ys311 = fit_weight33(xs)
ys322 = fit_weight_gra11(xss)
ys333 = fit_weight_gra22(xss)
ys344 = fit_weight_gra33(xss)




fitloss111 = make_interp_spline(x1,Totallossbox111)
fitloss222 = make_interp_spline(x1,Totallossbox222)
fitacc111 = make_interp_spline(x1,Accbox111)
fitacc222 = make_interp_spline(x1,Accbox222)
fitrec111_sam1_train = make_interp_spline(x1,rec111_sam_train[0])
fitrec111_sam2_train = make_interp_spline(x1,rec111_sam_train[1])
fitrec111_sam3_train = make_interp_spline(x1,rec111_sam_train[2])
fitpre111_sam1_train = make_interp_spline(x1,pre111_sam_train[0])
fitpre111_sam2_train = make_interp_spline(x1,pre111_sam_train[1])
fitpre111_sam3_train = make_interp_spline(x1,pre111_sam_train[2])
fitrec111_sam1_test = make_interp_spline(x1,rec111_sam_test[0])
fitrec111_sam2_test = make_interp_spline(x1,rec111_sam_test[1])
fitrec111_sam3_test = make_interp_spline(x1,rec111_sam_test[2])
fitpre111_sam1_test = make_interp_spline(x1,pre111_sam_test[0])
fitpre111_sam2_test = make_interp_spline(x1,pre111_sam_test[1])
fitpre111_sam3_test = make_interp_spline(x1,pre111_sam_test[2])
fit_f111_train = make_interp_spline(x1,f111[0])
fit_f111_test = make_interp_spline(x1,f111[1])
fit_f222_train = make_interp_spline(x1,f222[0])
fit_f222_test = make_interp_spline(x1,f222[1])
fit_f333_train = make_interp_spline(x1,f333[0])
fit_f333_test = make_interp_spline(x1,f333[1])
fit_weight111 = make_interp_spline(x1,weight111[0])
fit_weight222 = make_interp_spline(x1,weight111[1])
fit_weight333 = make_interp_spline(x1,weight111[2])
fit_weight_gra111 = make_interp_spline(x2,weight111_gra[0])
fit_weight_gra222 = make_interp_spline(x2,weight111_gra[1])
fit_weight_gra333 = make_interp_spline(x2,weight111_gra[2])
ys111 = fitloss111(xs)
ys222 = fitloss222(xs)
ys333 = fitacc111(xs)
ys444 = fitacc222(xs)
ys555 = fitrec111_sam1_train(xs)
ys666 = fitrec111_sam2_train(xs)
ys777 = fitrec111_sam3_train(xs)
ys999 = fitpre111_sam1_train(xs)
ys1000 = fitpre111_sam2_train(xs)
ys1111 = fitpre111_sam3_train(xs)
ys1333 = fitrec111_sam1_test(xs)
ys1444 = fitrec111_sam2_test(xs)
ys1555 = fitrec111_sam3_test(xs)
ys1777 = fitpre111_sam1_test(xs)
ys1888 = fitpre111_sam2_test(xs)
ys1999 = fitpre111_sam3_test(xs)
ys2111 = fit_f111_train(xs)
ys2222 = fit_f111_test(xs)
ys2333 = fit_f222_train(xs)
ys2444 = fit_f222_test(xs)
ys2555 = fit_f333_train(xs)
ys2666 = fit_f333_test(xs)
ys2999 = fit_weight111(xs)
ys3000 = fit_weight222(xs)
ys3111 = fit_weight333(xs)
ys3222 = fit_weight_gra111(xss)
ys3333 = fit_weight_gra222(xss)
ys3444 = fit_weight_gra333(xss)




fitloss111e = make_interp_spline(x1,Totallossbox1111)
fitloss222e = make_interp_spline(x1,Totallossbox2222)
fitacc111e = make_interp_spline(x1,Accbox1111)
fitacc222e = make_interp_spline(x1,Accbox2222)
fitrec111e_sam1_train = make_interp_spline(x1,rec1111_sam_train[0])
fitrec111e_sam2_train = make_interp_spline(x1,rec1111_sam_train[1])
fitrec111e_sam3_train = make_interp_spline(x1,rec1111_sam_train[2])
fitpre111e_sam1_train = make_interp_spline(x1,pre1111_sam_train[0])
fitpre111e_sam2_train = make_interp_spline(x1,pre1111_sam_train[1])
fitpre111e_sam3_train = make_interp_spline(x1,pre1111_sam_train[2])
fitrec111e_sam1_test = make_interp_spline(x1,rec1111_sam_test[0])
fitrec111e_sam2_test = make_interp_spline(x1,rec1111_sam_test[1])
fitrec111e_sam3_test = make_interp_spline(x1,rec1111_sam_test[2])
fitpre111e_sam1_test = make_interp_spline(x1,pre1111_sam_test[0])
fitpre111e_sam2_test = make_interp_spline(x1,pre1111_sam_test[1])
fitpre111e_sam3_test = make_interp_spline(x1,pre1111_sam_test[2])
fit_f111e_train = make_interp_spline(x1,f1111[0])
fit_f111e_test = make_interp_spline(x1,f1111[1])
fit_f222e_train = make_interp_spline(x1,f2222[0])
fit_f222e_test = make_interp_spline(x1,f2222[1])
fit_f333e_train = make_interp_spline(x1,f3333[0])
fit_f333e_test = make_interp_spline(x1,f3333[1])
fit_weight111e = make_interp_spline(x1,weight1111[0])
fit_weight222e = make_interp_spline(x1,weight1111[1])
fit_weight333e = make_interp_spline(x1,weight1111[2])
fit_weight_gra111e = make_interp_spline(x2,weight1111_gra[0])
fit_weight_gra222e = make_interp_spline(x2,weight1111_gra[1])
fit_weight_gra333e = make_interp_spline(x2,weight1111_gra[2])
ys111e = fitloss111e(xs)
ys222e = fitloss222e(xs)
ys333e = fitacc111e(xs)
ys444e = fitacc222e(xs)
ys555e = fitrec111e_sam1_train(xs)
ys666e = fitrec111e_sam2_train(xs)
ys777e = fitrec111e_sam3_train(xs)
ys999e = fitpre111e_sam1_train(xs)
ys1000e = fitpre111e_sam2_train(xs)
ys1111e = fitpre111e_sam3_train(xs)
ys1333e = fitrec111e_sam1_test(xs)
ys1444e = fitrec111e_sam2_test(xs)
ys1555e = fitrec111e_sam3_test(xs)
ys1777e = fitpre111e_sam1_test(xs)
ys1888e = fitpre111e_sam2_test(xs)
ys1999e = fitpre111e_sam3_test(xs)
ys2111e = fit_f111e_train(xs)
ys2222e = fit_f111e_test(xs)
ys2333e = fit_f222e_train(xs)
ys2444e = fit_f222e_test(xs)
ys2555e = fit_f333e_train(xs)
ys2666e = fit_f333e_test(xs)
ys2999e = fit_weight111e(xs)
ys3000e = fit_weight222e(xs)
ys3111e = fit_weight333e(xs)
ys3222e = fit_weight_gra111e(xss)
ys3333e = fit_weight_gra222e(xss)
ys3444e = fit_weight_gra333e(xss)

In [82]:
sum1 = np.zeros(28)
a1 = [[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[]] #28个

for i in range(200):
    sum1[0] += rec1_sam_train[0][i+100]
    sum1[1] += rec1_sam_train[1][i+100]
    sum1[2] += rec1_sam_train[2][i+100]
    sum1[4] += pre1_sam_train[0][i+100]
    sum1[5] += pre1_sam_train[1][i+100]
    sum1[6] += pre1_sam_train[2][i+100]
    sum1[8] += rec1_sam_test[0][i+100]
    sum1[9] += rec1_sam_test[1][i+100]
    sum1[10] += rec1_sam_test[2][i+100]
    sum1[12] += pre1_sam_test[0][i+100]
    sum1[13] += pre1_sam_test[1][i+100]
    sum1[14] += pre1_sam_test[2][i+100]
    sum1[16] += f1[0][i+100]
    sum1[17] += f1[1][i+100]
    sum1[18] += f2[0][i+100]
    sum1[19] += f2[1][i+100]
    sum1[20] += f3[0][i+100]
    sum1[21] += f3[1][i+100]
    sum1[24] += Totallossbox1[i+100]
    sum1[25] += Totallossbox2[i+100]
    sum1[26] += Accbox1[i+100]
    sum1[27] += Accbox2[i+100]
    
    a1[0] = np.append(a1[0],rec1_sam_train[0][i+100])
    a1[1] = np.append(a1[1],rec1_sam_train[1][i+100])
    a1[2] = np.append(a1[2],rec1_sam_train[2][i+100])
    a1[4] = np.append(a1[4],pre1_sam_train[0][i+100])
    a1[5] = np.append(a1[5],pre1_sam_train[1][i+100])
    a1[6] = np.append(a1[6],pre1_sam_train[2][i+100])
    a1[8] = np.append(a1[8],rec1_sam_test[0][i+100])
    a1[9] = np.append(a1[9],rec1_sam_test[1][i+100])
    a1[10] = np.append(a1[10],rec1_sam_test[2][i+100])
    a1[12] = np.append(a1[12],pre1_sam_test[0][i+100])
    a1[13] = np.append(a1[13],pre1_sam_test[1][i+100])
    a1[14] = np.append(a1[14],pre1_sam_test[2][i+100])
    a1[16] = np.append(a1[16],f1[0][i+100])
    a1[17] = np.append(a1[17],f1[1][i+100])
    a1[18] = np.append(a1[18],f2[0][i+100])
    a1[19] = np.append(a1[19],f2[1][i+100])
    a1[20] = np.append(a1[20],f3[0][i+100])
    a1[21] = np.append(a1[21],f3[1][i+100])
    a1[24] = np.append(a1[24],Totallossbox1[i+100])
    a1[25] = np.append(a1[25],Totallossbox2[i+100])
    a1[26] = np.append(a1[26],Accbox1[i+100])
    a1[27] = np.append(a1[27],Accbox2[i+100])
for j in range(28):
    a1[j] = np.var(a1[j])
       
for k in range(28):
    sum1[k] = sum1[k]/200 

D:\anaconda3\lib\site-packages\numpy\core\fromnumeric.py:3723: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
D:\anaconda3\lib\site-packages\numpy\core\_methods.py:222: RuntimeWarning: invalid value encountered in true_divide
  arrmean = um.true_divide(arrmean, div, out=arrmean, casting='unsafe',
D:\anaconda3\lib\site-packages\numpy\core\_methods.py:254: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


In [83]:
sum11 = np.zeros(28)
a11 = [[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[]] #28个

for i in range(200):
    sum11[0] += rec11_sam_train[0][i+100]
    sum11[1] += rec11_sam_train[1][i+100]
    sum11[2] += rec11_sam_train[2][i+100]
    sum11[4] += pre11_sam_train[0][i+100]
    sum11[5] += pre11_sam_train[1][i+100]
    sum11[6] += pre11_sam_train[2][i+100]
    sum11[8] += rec11_sam_test[0][i+100]
    sum11[9] += rec11_sam_test[1][i+100]
    sum11[10] += rec11_sam_test[2][i+100]
    sum11[12] += pre11_sam_test[0][i+100]
    sum11[13] += pre11_sam_test[1][i+100]
    sum11[14] += pre11_sam_test[2][i+100]
    sum11[16] += f11[0][i+100]
    sum11[17] += f11[1][i+100]
    sum11[18] += f22[0][i+100]
    sum11[19] += f22[1][i+100]
    sum11[20] += f33[0][i+100]
    sum11[21] += f33[1][i+100]
    sum11[24] += Totallossbox11[i+100]
    sum11[25] += Totallossbox22[i+100]
    sum11[26] += Accbox11[i+100]
    sum11[27] += Accbox22[i+100]
    
    a11[0] = np.append(a11[0],rec11_sam_train[0][i+100])
    a11[1] = np.append(a11[1],rec11_sam_train[1][i+100])
    a11[2] = np.append(a11[2],rec11_sam_train[2][i+100])
    a11[4] = np.append(a11[4],pre11_sam_train[0][i+100])
    a11[5] = np.append(a11[5],pre11_sam_train[1][i+100])
    a11[6] = np.append(a11[6],pre11_sam_train[2][i+100])
    a11[8] = np.append(a11[8],rec11_sam_test[0][i+100])
    a11[9] = np.append(a11[9],rec11_sam_test[1][i+100])
    a11[10] = np.append(a11[10],rec11_sam_test[2][i+100])
    a11[12] = np.append(a11[12],pre11_sam_test[0][i+100])
    a11[13] = np.append(a11[13],pre11_sam_test[1][i+100])
    a11[14] = np.append(a11[14],pre11_sam_test[2][i+100])
    a11[16] = np.append(a11[16],f11[0][i+100])
    a11[17] = np.append(a11[17],f11[1][i+100])
    a11[18] = np.append(a11[18],f22[0][i+100])
    a11[19] = np.append(a11[19],f22[1][i+100])
    a11[20] = np.append(a11[20],f33[0][i+100])
    a11[21] = np.append(a11[21],f33[1][i+100])
    a11[24] = np.append(a11[24],Totallossbox11[i+100])
    a11[25] = np.append(a11[25],Totallossbox22[i+100])
    a11[26] = np.append(a11[26],Accbox11[i+100])
    a11[27] = np.append(a11[27],Accbox22[i+100]) 
for j in range(28):
    a11[j] = np.var(a11[j])   
for k in range(28):
    sum11[k] = sum11[k]/200

In [84]:
sum111 = np.zeros(28)
a111 = [[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[]] #28个

for i in range(200):
    sum111[0] += rec111_sam_train[0][i+100]
    sum111[1] += rec111_sam_train[1][i+100]
    sum111[2] += rec111_sam_train[2][i+100]
    sum111[4] += pre111_sam_train[0][i+100]
    sum111[5] += pre111_sam_train[1][i+100]
    sum111[6] += pre111_sam_train[2][i+100]
    sum111[8] += rec111_sam_test[0][i+100]
    sum111[9] += rec111_sam_test[1][i+100]
    sum111[10] += rec111_sam_test[2][i+100]
    sum111[12] += pre111_sam_test[0][i+100]
    sum111[13] += pre111_sam_test[1][i+100]
    sum111[14] += pre111_sam_test[2][i+100]
    sum111[16] += f111[0][i+100]
    sum111[17] += f111[1][i+100]
    sum111[18] += f222[0][i+100]
    sum111[19] += f222[1][i+100]
    sum111[20] += f333[0][i+100]
    sum111[21] += f333[1][i+100]
    sum111[24] += Totallossbox111[i+100]
    sum111[25] += Totallossbox222[i+100]
    sum111[26] += Accbox111[i+100]
    sum111[27] += Accbox222[i+100]
    
    a111[0] = np.append(a111[0],rec111_sam_train[0][i+100])
    a111[1] = np.append(a111[1],rec111_sam_train[1][i+100])
    a111[2] = np.append(a111[2],rec111_sam_train[2][i+100])
    a111[4] = np.append(a111[4],pre111_sam_train[0][i+100])
    a111[5] = np.append(a111[5],pre111_sam_train[1][i+100])
    a111[6] = np.append(a111[6],pre111_sam_train[2][i+100])
    a111[8] = np.append(a111[8],rec111_sam_test[0][i+100])
    a111[9] = np.append(a111[9],rec111_sam_test[1][i+100])
    a111[10] = np.append(a111[10],rec111_sam_test[2][i+100])
    a111[12] = np.append(a111[12],pre111_sam_test[0][i+100])
    a111[13] = np.append(a111[13],pre111_sam_test[1][i+100])
    a111[14] = np.append(a111[14],pre111_sam_test[2][i+100])
    a111[16] = np.append(a111[16],f111[0][i+100])
    a111[17] = np.append(a111[17],f111[1][i+100])
    a111[18] = np.append(a111[18],f222[0][i+100])
    a111[19] = np.append(a111[19],f222[1][i+100])
    a111[20] = np.append(a111[20],f333[0][i+100])
    a111[21] = np.append(a111[21],f333[1][i+100])
    a111[24] = np.append(a111[24],Totallossbox111[i+100])
    a111[25] = np.append(a111[25],Totallossbox222[i+100])
    a111[26] = np.append(a111[26],Accbox111[i+100])
    a111[27] = np.append(a111[27],Accbox222[i+100])   
for j in range(28):
    a111[j] = np.var(a111[j])
for k in range(28):
    sum111[k] = sum111[k]/200


In [85]:
sum111e = np.zeros(28)
a111e = [[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[]] #28个

for i in range(200):
    sum111e[0] += rec1111_sam_train[0][i+100]
    sum111e[1] += rec1111_sam_train[1][i+100]
    sum111e[2] += rec1111_sam_train[2][i+100]
    sum111e[4] += pre1111_sam_train[0][i+100]
    sum111e[5] += pre1111_sam_train[1][i+100]
    sum111e[6] += pre1111_sam_train[2][i+100]
    sum111e[8] += rec1111_sam_test[0][i+100]
    sum111e[9] += rec1111_sam_test[1][i+100]
    sum111e[10] += rec1111_sam_test[2][i+100]
    sum111e[12] += pre1111_sam_test[0][i+100]
    sum111e[13] += pre1111_sam_test[1][i+100]
    sum111e[14] += pre1111_sam_test[2][i+100]
    sum111e[16] += f1111[0][i+100]
    sum111e[17] += f1111[1][i+100]
    sum111e[18] += f2222[0][i+100]
    sum111e[19] += f2222[1][i+100]
    sum111e[20] += f3333[0][i+100]
    sum111e[21] += f3333[1][i+100]
    sum111e[24] += Totallossbox1111[i+100]
    sum111e[25] += Totallossbox2222[i+100]
    sum111e[26] += Accbox1111[i+100]
    sum111e[27] += Accbox2222[i+100]
    
    a111e[0] = np.append(a111e[0],rec1111_sam_train[0][i+100])
    a111e[1] = np.append(a111e[1],rec1111_sam_train[1][i+100])
    a111e[2] = np.append(a111e[2],rec1111_sam_train[2][i+100])
    a111e[4] = np.append(a111e[4],pre1111_sam_train[0][i+100])
    a111e[5] = np.append(a111e[5],pre1111_sam_train[1][i+100])
    a111e[6] = np.append(a111e[6],pre1111_sam_train[2][i+100])
    a111e[8] = np.append(a111e[8],rec1111_sam_test[0][i+100])
    a111e[9] = np.append(a111e[9],rec1111_sam_test[1][i+100])
    a111e[10] = np.append(a111e[10],rec1111_sam_test[2][i+100])
    a111e[12] = np.append(a111e[12],pre1111_sam_test[0][i+100])
    a111e[13] = np.append(a111e[13],pre1111_sam_test[1][i+100])
    a111e[14] = np.append(a111e[14],pre1111_sam_test[2][i+100])
    a111e[16] = np.append(a111e[16],f1111[0][i+100])
    a111e[17] = np.append(a111e[17],f1111[1][i+100])
    a111e[18] = np.append(a111e[18],f2222[0][i+100])
    a111e[19] = np.append(a111e[19],f2222[1][i+100])
    a111e[20] = np.append(a111e[20],f3333[0][i+100])
    a111e[21] = np.append(a111e[21],f3333[1][i+100])
    a111e[24] = np.append(a111e[24],Totallossbox1111[i+100])
    a111e[25] = np.append(a111e[25],Totallossbox2222[i+100])
    a111e[26] = np.append(a111e[26],Accbox1111[i+100])
    a111e[27] = np.append(a111e[27],Accbox2222[i+100])
    
for j in range(28):
    a111e[j] = np.var(a111e[j])
for k in range(28):
    sum111e[k] = sum111e[k]/200

In [86]:
with open('iris NN.txt', 'a') as f:
    lines = ['4种神经网络P-R曲线面积为:',' ',str(pr_aucFNN1),',',' ',str(pr_aucQRFNN),',',' ',str(pr_aucFNN2),',',' ',str(pr_aucQDFNN),'\n'
             '4种神经网络ROC曲线面积为:',' ',str(roc_aucFNN1),',',' ',str(roc_aucQRFNN),',',' ',str(roc_aucFNN2),',',' ',str(roc_aucQDFNN),'\n','\n',
             '第一组神经网络的样本0train的recall方差:',' ',str(a1[0]),'\n',
             '第一组神经网络的样本1train的recall方差:',' ',str(a1[1]),'\n',
             '第一组神经网络的样本2train的recall方差:',' ',str(a1[2]),'\n',
             '第一组神经网络的样本0train的precision方差:',' ',str(a1[4]),'\n',
             '第一组神经网络的样本1train的precision方差:',' ',str(a1[5]),'\n',
             '第一组神经网络的样本2train的precision方差:',' ',str(a1[6]),'\n',
             '第一组神经网络的样本0test的recall方差:',' ',str(a1[8]),'\n',
             '第一组神经网络的样本1test的recall方差:',' ',str(a1[9]),'\n',
             '第一组神经网络的样本2test的recall方差:',' ',str(a1[10]),'\n',
             '第一组神经网络的样本0test的precision方差:',' ',str(a1[12]),'\n',
             '第一组神经网络的样本1test的precision方差:',' ',str(a1[13]),'\n',
             '第一组神经网络的样本2test的precision方差:',' ',str(a1[14]),'\n',
             '第一组神经网络的样本0train的F1-score方差:',' ',str(a1[16]),'\n',
             '第一组神经网络的样本0test的F1-score方差:',' ',str(a1[17]),'\n',
             '第一组神经网络的样本1train的F1-score方差:',' ',str(a1[18]),'\n',
             '第一组神经网络的样本1test的F1-score方差:',' ',str(a1[19]),'\n',
             '第一组神经网络的样本2train的F1-score方差:',' ',str(a1[20]),'\n',
             '第一组神经网络的样本2test的F1-score方差:',' ',str(a1[21]),'\n',
             "第一组神经网络的loss的train方差:",' ',str(a1[24]),'\n',
             "第一组神经网络的loss的test方差:",' ',str(a1[25]),'\n',
             "第一组神经网络的acc的train方差:",' ',str(a1[26]),'\n',
             "第一组神经网络的acc的test方差:",' ',str(a1[27]),'\n',
             
             '第一组神经网络的样本0train的recall均值:',' ',str(sum1[0]),'\n',
            '第一组神经网络的样本1train的recall均值:',' ',str(sum1[1]),'\n',
            '第一组神经网络的样本2train的recall均值:',' ',str(sum1[2]),'\n',
            '第一组神经网络的样本0train的precision均值:',' ',str(sum1[4]),'\n',
            '第一组神经网络的样本1train的precision均值:',' ',str(sum1[5]),'\n',
            '第一组神经网络的样本2train的precision均值:',' ',str(sum1[6]),'\n',
            '第一组神经网络的样本0test的recall均值:',' ',str(sum1[8]),'\n',
            '第一组神经网络的样本1test的recall均值:',' ',str(sum1[9]),'\n',
            '第一组神经网络的样本2test的recall均值:',' ',str(sum1[10]),'\n',
            '第一组神经网络的样本0test的precision均值:',' ',str(sum1[12]),'\n',
            '第一组神经网络的样本1test的precision均值:',' ',str(sum1[13]),'\n',
            '第一组神经网络的样本2test的precision均值:',' ',str(sum1[14]),'\n',
            '第一组神经网络的样本0train的F1-score均值:',' ',str(sum1[16]),'\n',
            '第一组神经网络的样本0test的F1-score均值:',' ',str(sum1[17]),'\n',
            '第一组神经网络的样本1train的F1-score均值:',' ',str(sum1[18]),'\n',
            '第一组神经网络的样本1test的F1-score均值:',' ',str(sum1[19]),'\n',
            '第一组神经网络的样本2train的F1-score均值:',' ',str(sum1[20]),'\n',
            '第一组神经网络的样本2test的F1-score均值:',' ',str(sum1[21]),'\n',
            "第一组神经网络的loss的train均值:",' ',str(sum1[24]),'\n',
            "第一组神经网络的loss的test均值:",' ',str(sum1[25]),'\n',
            "第一组神经网络的acc的train均值:",' ',str(sum1[26]),'\n',
            "第一组神经网络的acc的test均值:",' ',str(sum1[27]),'\n',
             
             '第2组神经网络的样本0train的recall方差:',' ',str(a11[0]),'\n',   
            '第2组神经网络的样本1train的recall方差:',' ',str(a11[1]),'\n', 
            '第2组神经网络的样本2train的recall方差:',' ',str(a11[2]),'\n', 
            '第2组神经网络的样本0train的precision方差:',' ',str(a11[4]),'\n', 
            '第2组神经网络的样本1train的precision方差:',' ',str(a11[5]),'\n', 
            '第2组神经网络的样本2train的precision方差:',' ',str(a11[6]),'\n', 
            '第2组神经网络的样本0test的recall方差:',' ',str(a11[8]),'\n', 
            '第2组神经网络的样本1test的recall方差:',' ',str(a11[9]),'\n', 
            '第2组神经网络的样本2test的recall方差:',' ',str(a11[10]),'\n', 
            '第2组神经网络的样本0test的precision方差:',' ',str(a11[12]),'\n', 
            '第2组神经网络的样本1test的precision方差:',' ',str(a11[13]),'\n', 
            '第2组神经网络的样本2test的precision方差:',' ',str(a11[14]),'\n',  
            '第2组神经网络的样本0train的F1-score方差:',' ',str(a11[16]),'\n', 
            '第2组神经网络的样本0test的F1-score方差:',' ',str(a11[17]),'\n', 
            '第2组神经网络的样本1train的F1-score方差:',' ',str(a11[18]),'\n', 
            '第2组神经网络的样本1test的F1-score方差:',' ',str(a11[19]),'\n', 
            '第2组神经网络的样本2train的F1-score方差:',' ',str(a11[20]),'\n', 
            '第2组神经网络的样本2test的F1-score方差:',' ',str(a11[21]),'\n', 
            "第2组神经网络的loss的train方差:",' ',str(a11[24]),'\n', 
            "第2组神经网络的loss的test方差:",' ',str(a11[25]),'\n', 
            "第2组神经网络的acc的train方差:",' ',str(a11[26]),'\n', 
            "第2组神经网络的acc的test方差:",' ',str(a11[27]),'\n', 
             
             '第2组神经网络的样本0train的recall均值:',' ',str(sum11[0]),'\n',  
            '第2组神经网络的样本1train的recall均值:',' ',str(sum11[1]),'\n',  
            '第2组神经网络的样本2train的recall均值:',' ',str(sum11[2]),'\n',   
            '第2组神经网络的样本0train的precision均值:',' ',str(sum11[4]),'\n',  
            '第2组神经网络的样本1train的precision均值:',' ',str(sum11[5]),'\n',  
            '第2组神经网络的样本2train的precision均值:',' ',str(sum11[6]),'\n',  
            '第2组神经网络的样本0test的recall均值:',' ',str(sum11[8]),'\n',  
            '第2组神经网络的样本1test的recall均值:',' ',str(sum11[9]),'\n',  
            '第2组神经网络的样本2test的recall均值:',' ',str(sum11[10]),'\n',    
            '第2组神经网络的样本0test的precision均值:',' ',str(sum11[12]),'\n',  
            '第2组神经网络的样本1test的precision均值:',' ',str(sum11[13]),'\n',  
            '第2组神经网络的样本2test的precision均值:',' ',str(sum11[14]),'\n',   
            '第2组神经网络的样本0train的F1-score均值:',' ',str(sum11[16]),'\n',  
            '第2组神经网络的样本0test的F1-score均值:',' ',str(sum11[17]),'\n',  
            '第2组神经网络的样本1train的F1-score均值:',' ',str(sum11[18]),'\n',  
            '第2组神经网络的样本1test的F1-score均值:',' ',str(sum11[19]),'\n',  
            '第2组神经网络的样本2train的F1-score均值:',' ',str(sum11[20]),'\n',  
            '第2组神经网络的样本2test的F1-score均值:',' ',str(sum11[21]),'\n',  
            "第2组神经网络的loss的train均值:",' ',str(sum11[24]),'\n',  
            "第2组神经网络的loss的test均值:",' ',str(sum11[25]),'\n',  
            "第2组神经网络的acc的train均值:",' ',str(sum11[26]),'\n',  
            "第2组神经网络的acc的test均值:",' ',str(sum11[27]),'\n', 
             
             '第3组神经网络的样本0train的recall方差:',' ',str(a111[0]),'\n',
            '第3组神经网络的样本1train的recall方差:',' ',str(a111[1]),'\n',
            '第3组神经网络的样本2train的recall方差:',' ',str(a111[2]),'\n',
            '第3组神经网络的样本0train的precision方差:',' ',str(a111[4]),'\n',
            '第3组神经网络的样本1train的precision方差:',' ',str(a111[5]),'\n',
            '第3组神经网络的样本2train的precision方差:',' ',str(a111[6]),'\n',
            '第3组神经网络的样本0test的recall方差:',' ',str(a111[8]),'\n',
            '第3组神经网络的样本1test的recall方差:',' ',str(a111[9]),'\n',
            '第3组神经网络的样本2test的recall方差:',' ',str(a111[10]),'\n',
            '第3组神经网络的样本0test的precision方差:',' ',str(a111[12]),'\n',
            '第3组神经网络的样本1test的precision方差:',' ',str(a111[13]),'\n',
            '第3组神经网络的样本2test的precision方差:',' ',str(a111[14]),'\n',
            '第3组神经网络的样本0train的F1-score方差:',' ',str(a111[16]),'\n',
            '第3组神经网络的样本0test的F1-score方差:',' ',str(a111[17]),'\n',
            '第3组神经网络的样本1train的F1-score方差:',' ',str(a111[18]),'\n',
            '第3组神经网络的样本1test的F1-score方差:',' ',str(a111[19]),'\n',
            '第3组神经网络的样本2train的F1-score方差:',' ',str(a111[20]),'\n',
            '第3组神经网络的样本2test的F1-score方差:',' ',str(a111[21]),'\n',
            "第3组神经网络的loss的train方差:",' ',str(a111[24]),'\n',
            "第3组神经网络的loss的test方差:",' ',str(a111[25]),'\n',
            "第3组神经网络的acc的train方差:",' ',str(a111[26]),'\n',
            "第3组神经网络的acc的test方差:",' ',str(a111[27]),'\n',
             
             '第3组神经网络的样本0train的recall均值:',' ',str(sum111[0]),'\n',
            '第3组神经网络的样本1train的recall均值:',' ',str(sum111[1]),'\n',
            '第3组神经网络的样本2train的recall均值:',' ',str(sum111[2]),'\n',
            '第3组神经网络的样本0train的precision均值:',' ',str(sum111[4]),'\n',
            '第3组神经网络的样本1train的precision均值:',' ',str(sum111[5]),'\n',
            '第3组神经网络的样本2train的precision均值:',' ',str(sum111[6]),'\n',
            '第3组神经网络的样本0test的recall均值:',' ',str(sum111[8]),'\n',
            '第3组神经网络的样本1test的recall均值:',' ',str(sum111[9]),'\n',
            '第3组神经网络的样本2test的recall均值:',' ',str(sum111[10]),'\n',
            '第3组神经网络的样本0test的precision均值:',' ',str(sum111[12]),'\n',
            '第3组神经网络的样本1test的precision均值:',' ',str(sum111[13]),'\n',
            '第3组神经网络的样本2test的precision均值:',' ',str(sum111[14]),'\n',
            '第3组神经网络的样本0train的F1-score均值:',' ',str(sum111[16]),'\n',
            '第3组神经网络的样本0test的F1-score均值:',' ',str(sum111[17]),'\n',
            '第3组神经网络的样本1train的F1-score均值:',' ',str(sum111[18]),'\n',
            '第3组神经网络的样本2test的F1-score均值:',' ',str(sum111[19]),'\n',
            '第3组神经网络的样本1train的F1-score均值:',' ',str(sum111[20]),'\n',
            '第3组神经网络的样本2test的F1-score均值:',' ',str(sum111[21]),'\n',
            "第3组神经网络的loss的train均值:",' ',str(sum111[24]),'\n',
            "第3组神经网络的loss的test均值:",' ',str(sum111[25]),'\n',
            "第3组神经网络的acc的train均值:",' ',str(sum111[26]),'\n',
            "第3组神经网络的acc的test均值:",' ',str(sum111[27]),'\n',
             
            '第4组神经网络的样本0train的recall方差:',' ',str(a111e[0]),'\n',
            '第4组神经网络的样本1train的recall方差:',' ',str(a111e[1]),'\n',
            '第4组神经网络的样本2train的recall方差:',' ',str(a111e[2]),'\n',
            '第4组神经网络的样本0train的precision方差:',' ',str(a111e[4]),'\n',
            '第4组神经网络的样本1train的precision方差:',' ',str(a111e[5]),'\n',
            '第4组神经网络的样本2train的precision方差:',' ',str(a111e[6]),'\n',
            '第4组神经网络的样本0test的recall方差:',' ',str(a111e[8]),'\n',
            '第4组神经网络的样本1test的recall方差:',' ',str(a111e[9]),'\n',
            '第4组神经网络的样本2test的recall方差:',' ',str(a111e[10]),'\n',
            '第4组神经网络的样本0test的precision方差:',' ',str(a111e[12]),'\n',
            '第4组神经网络的样本1test的precision方差:',' ',str(a111e[13]),'\n',
            '第4组神经网络的样本2test的precision方差:',' ',str(a111e[14]),'\n',
            '第4组神经网络的样本0train的F1-score方差:',' ',str(a111e[16]),'\n',
            '第4组神经网络的样本0test的F1-score方差:',' ',str(a111e[17]),'\n',
            '第4组神经网络的样本1train的F1-score方差:',' ',str(a111e[18]),'\n',
            '第4组神经网络的样本1test的F1-score方差:',' ',str(a111e[19]),'\n',
            '第4组神经网络的样本2train的F1-score方差:',' ',str(a111e[20]),'\n',
            '第4组神经网络的样本2test的F1-score方差:',' ',str(a111e[21]),'\n',
            "第4组神经网络的loss的train方差:",' ',str(a111e[24]),'\n',
            "第4组神经网络的loss的test方差:",' ',str(a111e[25]),'\n',
            "第4组神经网络的acc的train方差:",' ',str(a111e[26]),'\n',
            "第4组神经网络的acc的test方差:",' ',str(a111e[27]),'\n',
             
             '第4组神经网络的样本0train的recall均值:',' ',str(sum111e[0]),'\n',
            '第4组神经网络的样本1train的recall均值:',' ',str(sum111e[1]),'\n',
            '第4组神经网络的样本2train的recall均值:',' ',str(sum111e[2]),'\n',
            '第4组神经网络的样本0train的precision均值:',' ',str(sum111e[4]),'\n',
            '第4组神经网络的样本1train的precision均值:',' ',str(sum111e[5]),'\n',
            '第4组神经网络的样本2train的precision均值:',' ',str(sum111e[6]),'\n',
            '第4组神经网络的样本0test的recall均值:',' ',str(sum111e[8]),'\n',
            '第4组神经网络的样本1test的recall均值:',' ',str(sum111e[9]),'\n',
            '第4组神经网络的样本2test的recall均值:',' ',str(sum111e[10]),'\n',
            '第4组神经网络的样本0test的precision均值:',' ',str(sum111e[12]),'\n',
            '第4组神经网络的样本1test的precision均值:',' ',str(sum111e[13]),'\n',
            '第4组神经网络的样本2test的precision均值:',' ',str(sum111e[14]),'\n',
            '第4组神经网络的样本0train的F1-score均值:',' ',str(sum111e[16]),'\n',
            '第4组神经网络的样本0test的F1-score均值:',' ',str(sum111e[17]),'\n',
            '第4组神经网络的样本1train的F1-score均值:',' ',str(sum111e[18]),'\n',
            '第4组神经网络的样本1test的F1-score均值:',' ',str(sum111e[19]),'\n',
            '第4组神经网络的样本2train的F1-score均值:',' ',str(sum111e[20]),'\n',
            '第4组神经网络的样本2test的F1-score均值:',' ',str(sum111e[21]),'\n',
            "第4组神经网络的loss的train均值:",' ',str(sum111e[24]),'\n',
            "第4组神经网络的loss的test均值:",' ',str(sum111e[25]),'\n',
            "第4组神经网络的acc的train均值:",' ',str(sum111e[26]),'\n',
            "第4组神经网络的acc的test均值:",' ',str(sum111e[27]),'\n',
            ]
    f.writelines(lines)

In [41]:
%matplotlib qt5
plt.subplot(121)
#plt.plot(xs,ys1,color = "blue",label = "training of FNN(rot-relu)",lw = 3)
#plt.plot(xs,ys2,color = "red",label = "testing of FNN(rot-relu)",lw = 3)
#plt.plot(xs,yss11,color = "green",label = "training of QRFNN",lw = 3)
#plt.plot(xs,yss22,color = "yellow",label = "testing of QRFNN",lw = 3)
#plt.plot(xs,ys111,color = "black",label = "training of FNN(leaky-relu)",lw = 3)
#plt.plot(xs,ys222,color = "orange",label = "testing of FNN(leaky-relu)",lw = 3)
#plt.plot(xs,ys111e,color = "grey",label = "training of QDFNN",lw = 3)
#plt.plot(xs,ys222e,color = "purple",label = "testing of QDFNN",lw = 3)

#plt.title("loss-epoch plot of iris set")#with Gaussian noise in the first layer(loc=0,scale=0.00005)
#plt.xlabel("epoch")
#plt.ylim(0,0.03)
#plt.ylabel("loss")
#plt.legend(loc='upper right')
#plt.grid()

#plt.subplot(122)
plt.plot(xs,ys3,color = "blue",label = "training of FNN(rot-relu)",lw = 3)
plt.plot(xs,ys4,color = "red",label = "testing of FNN(rot-relu)",lw = 3)
plt.plot(xs,ys33,color = "green",label = "training of QRFNN",lw = 3)
plt.plot(xs,ys44,color = "yellow",label = "testing of QRFNN",lw = 3)
plt.plot(xs,ys333,color = "black",label = "training of FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys444,color = "orange",label = "testing of FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys333e,color = "grey",label = "training of QDFNN",lw = 3)
plt.plot(xs,ys444e,color = "purple",label = "testing of QDFNN",lw = 3)
plt.title("accuracy-epoch plot of iris set")#with Gaussian noise in the first layer(loc=0,scale=0.00005)
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend(loc='lower right')
plt.grid()

In [120]:
%matplotlib qt5
plt.subplot(221)
plt.plot(xs,ys5,label = "sample 0 in training set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys13,label = "sample 0 in testing set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys55,label = "sample 0 in training set in QRFNN",lw = 3)
plt.plot(xs,ys133,label = "sample 0 in testing set in QRFNN",lw = 3)
plt.plot(xs,ys555,label = "sample 0 in training set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys1333,label = "sample 0 in testing set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys555e,label = "sample 0 in training set in QDFNN",lw = 3)
plt.plot(xs,ys1333e,label = "sample 0 in testing set in QDFNN",lw = 3)
plt.title("recall-epoch plot of iris set of sample 0")
plt.xlabel("epoch")
plt.ylabel("recall")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(222)
plt.plot(xs,ys6,label = "sample 1 in training set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys14,label = "sample 1 in testing set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys66,label = "sample 1 in training set in QRFNN",lw = 3)
plt.plot(xs,ys144,label = "sample 1 in testing set in QRFNN",lw = 3)
plt.plot(xs,ys666,label = "sample 1 in training set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys1444,label = "sample 1 in testing set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys666e,label = "sample 1 in training set in QDFNN",lw = 3)
plt.plot(xs,ys1444e,label = "sample 1 in testing set in QDFNN",lw = 3)
plt.title("recall-epoch plot of iris set of sample 1")
plt.xlabel("epoch")
plt.ylabel("recall")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(223)
plt.plot(xs,ys7,label = "sample 2 in training set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys15,label = "sample 2 in testing set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys77,label = "sample 2 in training set in QRFNN",lw = 3)
plt.plot(xs,ys155,label = "sample 2 in testing set in QRFNN",lw = 3)
plt.plot(xs,ys777,label = "sample 2 in training set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys1555,label = "sample 2 in testing set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys777e,label = "sample 2 in training set in QDFNN",lw = 3)
plt.plot(xs,ys1555e,label = "sample 2 in testing set in QDFNN",lw = 3)
plt.title("recall-epoch plot of iris set of sample 2")
plt.xlabel("epoch")
plt.ylabel("recall")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()


In [89]:
%matplotlib qt5
plt.subplot(221)
plt.plot(xs,ys9,label = "sample 0 in training set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys17,label = "sample 0 in testing set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys99,label = "sample 0 in training set in QRFNN",lw = 3)
plt.plot(xs,ys177,label = "sample 0 in testing set in QRFNN",lw = 3)
plt.plot(xs,ys999,label = "sample 0 in training set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys1777,label = "sample 0 in testing set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys999e,label = "sample 0 in training set in QDFNN",lw = 3)
plt.plot(xs,ys1777e,label = "sample 0 in testing set in QDFNN",lw = 3)
plt.title("precision-epoch plot of iris set of sample 0")
plt.xlabel("epoch")
plt.ylabel("precision")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(222)
plt.plot(xs,ys10,label = "sample 1 in training set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys18,label = "sample 1 in testing set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys100,label = "sample 1 in training set in QRFNN",lw = 3)
plt.plot(xs,ys188,label = "sample 1 in testing set in QRFNN",lw = 3)
plt.plot(xs,ys1000,label = "sample 1 in training set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys1888,label = "sample 1 in testing set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys1000e,label = "sample 1 in training set in QDFNN",lw = 3)
plt.plot(xs,ys1888e,label = "sample 1 in testing set in QDFNN",lw = 3)
plt.title("precision-epoch plot of iris set of sample 1")
plt.xlabel("epoch")
plt.ylabel("precision")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(223)
plt.plot(xs,ys11,label = "sample 2 in training set in FNN(rot-relu)",lw = 3) 
plt.plot(xs,ys19,label = "sample 2 in testing set in FNN(rot-relu)",lw = 3)
plt.plot(xs,yss111,label = "sample 2 in training set in QRFNN",lw = 3)    
plt.plot(xs,ys199,label = "sample 2 in testing set in QRFNN",lw = 3)
plt.plot(xs,ys1111,label = "sample 2 in training set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys1999,label = "sample 2 in testing set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys1111e,label = "sample 2 in training set in QDFNN",lw = 3)
plt.plot(xs,ys1999e,label = "sample 2 in testing set in QDFNN",lw = 3)
plt.title("precision-epoch plot of iris set of sample 2")
plt.xlabel("epoch")
plt.ylabel("precision")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()


In [90]:
%matplotlib qt5
plt.subplot(221)
plt.plot(xs,ys21,label = "sample 0 in training set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys22,label = "sample 0 in testing set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys211,label = "sample 0 in training set in QRFNN",lw = 3)   
plt.plot(xs,yss222,label = "sample 0 in testing set in QRFNN",lw = 3)
plt.plot(xs,ys2111,label = "sample 0 in training set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys2222,label = "sample 0 in testing set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys2111e,label = "sample 0 in training set in QDFNN",lw = 3)
plt.plot(xs,ys2222e,label = "sample 0 in testing set in QDFNN",lw = 3)
plt.title("F score-epoch plot of iris set of sample 0")
plt.xlabel("epoch")
plt.ylabel("F score")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(222)
plt.plot(xs,ys23,label = "sample 1 in training set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys24,label = "sample 1 in testing set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys233,label = "sample 1 in training set in QRFNN",lw = 3)
plt.plot(xs,ys244,label = "sample 1 in testing set in QRFNN",lw = 3)
plt.plot(xs,ys2333,label = "sample 1 in training set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys2444,label = "sample 1 in testing set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys2333e,label = "sample 1 in training set in QDFNN",lw = 3)
plt.plot(xs,ys2444e,label = "sample 1 in testing set in QDFNN",lw = 3)
plt.title("F score-epoch plot of iris set of sample 1")
plt.xlabel("epoch")
plt.ylabel("F score")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(223)
plt.plot(xs,ys25,label = "sample 2 in training set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys26,label = "sample 2 in testing set in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys255,label = "sample 2 in training set in QRFNN",lw = 3)
plt.plot(xs,ys266,label = "sample 2 in testing set in QRFNN",lw = 3)
plt.plot(xs,ys2555,label = "sample 2 in training set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys2666,label = "sample 2 in testing set in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys2555e,label = "sample 2 in training set in QDFNN",lw = 3)
plt.plot(xs,ys2666e,label = "sample 2 in testing set in QDFNN",lw = 3)
plt.title("F score-epoch plot of iris set of sample 2")
plt.xlabel("epoch")
plt.ylabel("F score")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

In [91]:
%matplotlib qt5
plt.subplot(221)
plt.plot(xs,ys29,label = "weight 1 in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys30,label = "weight 2 in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys31,label = "weight 3 in FNN(rot-relu)",lw = 3)
plt.plot(xs,ys299,label = "weight 1 in QRFNN",lw = 3)
plt.plot(xs,ys300,label = "weight 2 in QRFNN",lw = 3)
plt.plot(xs,ys311,label = "weight 3 in QRFNN",lw = 3)
plt.plot(xs,ys2999,label = "weight 1 in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys3000,label = "weight 2 in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys3111,label = "weight 3 in FNN(leaky-relu)",lw = 3)
plt.plot(xs,ys2999e,label = "weight 1 in QDFNN",lw = 3)
plt.plot(xs,ys3000e,label = "weight 2 in QDFNN",lw = 3)
plt.plot(xs,ys3111e,label = "weight 3 in QDFNN",lw = 3)
plt.title("weight parameter-epoch plot in 4 networks in training set")
plt.xlabel("epoch")
plt.style.use('classic')
plt.ylabel("weight parameter")
plt.legend(loc='lower left',prop={'size':9})
plt.grid()

plt.subplot(222)
plt.plot(xss,ys32,label = "weight 1 grad in FNN(rot-relu)",lw = 3)
plt.plot(xss,ys33,label = "weight 2 grad in FNN(rot-relu)",lw = 3)
plt.plot(xss,ys34,label = "weight 3 grad in FNN(rot-relu)",lw = 3)
plt.plot(xss,ys322,label = "weight 1 grad in QRFNN",lw = 3)
plt.plot(xss,ys333,label = "weight 2 grad in QRFNN",lw = 3)
plt.plot(xss,ys344,label = "weight 3 grad in QRFNN",lw = 3)
plt.plot(xss,ys3222,label = "weight 1 grad in FNN(leaky-relu)",lw = 3)
plt.plot(xss,ys3333,label = "weight 2 grad in FNN(leaky-relu)",lw = 3)
plt.plot(xss,ys3444,label = "weight 3 grad in FNN(leaky-relu)",lw = 3)
plt.plot(xss,ys3222e,label = "weight 1 grad in QDFNN",lw = 3)
plt.plot(xss,ys3333e,label = "weight 2 grad in QDFNN",lw = 3)
plt.plot(xss,ys3444e,label = "weight 3 grad in QDFNN",lw = 3)
plt.title("weight parameter gradient-epoch plot in 4 networks in training set")
plt.style.use('classic')
plt.xlabel("epoch")
plt.ylabel("weight parameter gradient")
plt.legend(loc='lower left',prop={'size':9})
plt.grid()